# GPU search vs OpenSearch on Amazon ESCI

This notebook answers four questions with **matched** comparisons: both systems get the same catalog, the same queries, the same metrics, and the same tuning budget.

| # | Experiment | What is matched |
|---|---|---|
| 1 | **Lexical quality** | GPU BM25F vs OpenSearch `title`, `cross_fields` and weighted concatenation. Both sides are tuned on the same validation queries. |
| 2 | **Full pipeline quality** | Retrieve 20,000 (lexical + dense), rank with a GBDT on the GPU, facets, pagination. Same request, same feature definitions, same ranker training. |
| 3 | **Latency and throughput** | Only configurations whose quality was measured in 1 and 2, at the same depth and with the same filters. |
| 4 | **Cost** | Cost per million queries at a fixed p99 latency target. |

Correctness checks run inline and stop the notebook if they fail: the GPU scorer against a reference implementation, metrics against hand-computed values, fusion and features against an independent implementation, facets against a brute-force count, filters, and pagination.

**How to run.** Choose a GPU runtime with high RAM (A100 or L4), then *Run all*. Set `DEV = True` first for a quick end-to-end pass on 200,000 products. Expensive steps (postings, embeddings, the OpenSearch index) are cached under `DATA_DIR`; point it at Google Drive to keep them across sessions.

**Run time at full scale:** about 1 to 1.5 hours, most of it the OpenSearch HNSW build and the pipeline runs.

## 0 · Setup

In [ ]:
%pip install -q snowballstemmer opensearch-py sentence-transformers xgboost requests
%pip install -q cuvs-cu12 --extra-index-url=https://pypi.nvidia.com

import collections, gc, hashlib, json, math, os, pickle, re, subprocess, sys, tarfile, time, urllib.request
from dataclasses import dataclass, field
from itertools import chain
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import snowballstemmer
import torch
import xgboost as xgb
from sklearn.feature_extraction.text import CountVectorizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
if IS_CUDA:
    import cupy as cp
pd.set_option("display.max_columns", 40, "display.width", 200, "display.float_format", lambda x: f"{x:,.4f}")
print("Device:", torch.cuda.get_device_name() if IS_CUDA else "CPU (speed results will not describe a GPU)")
print("CPU cores:", os.cpu_count())

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu13 26.6.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==13.*, but you have cuda-toolkit 12.9.2.0 which is incompatible.
cudf-cu13 26.6.0 requires cuda-toolkit[nvcc,nvrtc]==13.*, but you have cuda-toolkit 12.9.2.0 which is incompatible.
libnvforest-cu13 26.6.0 requires rapids-logger==0.2.*, but you have rapids-logger 0.3.0 which is incompatible.
libraft-cu13 26.6.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==13.*, but you have cuda-toolkit 12.9.2.0 which is incompatible.
libraft-cu13 26.6.0 requires rapids-logger==0.2.*, but you have rapids-logger 0.3.0 which is incompatible.
libcugraph-cu13 26.6.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse,nvrtc]==13.*, but you have cuda-toolkit 12.9.2.0 which is incompatible.
libcuvs-cu13 26.6.0 requires cuda-toolkit[cublas,curand,cus

In [ ]:
# ===================================== Settings =====================================
DEV = False                         # True: 200k products and fewer queries, for a quick full pass
SEED = 13
LOCALE = "us"                       # The analyzer is English.
DATA_DIR = Path("/content/esci")    # e.g. Path("/content/drive/MyDrive/esci") to keep caches
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ---- Data ----------------------------------------------------------------------------
MAX_DOCS = 200_000 if DEV else None         # Judged products are always kept.
VAL_FRACTION = 0.10                          # Share of training queries held out for selection
N_SELECT = 1000                              # Validation queries used to select settings (both systems)
GAIN = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}   # ESCI paper gains
LABEL_INT = {"E": 3, "S": 2, "C": 1, "I": 0}       # GBDT training labels

# ---- Lexical ---------------------------------------------------------------------------
FIELDS = ["title", "brand", "color", "bullet_point", "description"]
COLUMN_OF = {"title": "product_title", "brand": "product_brand", "color": "product_color",
             "bullet_point": "product_bullet_point", "description": "product_description"}
BM25_K1, BM25_B = 1.2, 0.75                  # Lucene defaults
HAND_WEIGHTS = {"title": 3.0, "brand": 2.0, "color": 1.0, "bullet_point": 1.0, "description": 0.5}
GRAD_STEPS, GRAD_LR, SELECT_EVERY = (100 if DEV else 300), 0.05, 10
N_OS_TRIALS = 8 if DEV else 30               # Random-search trials for OpenSearch boosts
WCONCAT_MAX_REPS = 8                         # Weighted concatenation: most a field is repeated
EVAL_DEPTH, EVAL_BATCH = 1000, 64
MAX_POSTINGS_PER_CHUNK = 30_000_000

# ---- Dense and pipeline -----------------------------------------------------------------
EMBED_MODEL = "BAAI/bge-small-en-v1.5"       # Swap in DenseOn here and set QUERY_PREFIX to match.
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
EMBED_MAX_LEN, EMBED_BATCH = 64, 1024
DENSE_BACKEND = "cuvs_brute_force"           # "cuvs_brute_force" (exact), "cuvs_cagra", or "torch"
K_LEX, K_DENSE, K_CAND, K_FINAL = 20_000, 5_000, 20_000, 1_000
RRF_K = 60
PAGE_SIZE, N_PAGES = 24, 5
FACET_FIELDS, FACET_SIZE = ("brand", "color"), 20
KEYWORD_MAX_CHARS = 256                      # OpenSearch keyword ignore_above, mirrored on the GPU

# ---- Ranker -------------------------------------------------------------------------------
N_TRAIN_QUERIES, N_VAL_QUERIES, N_TEST_QUERIES = (600, 200, 500) if DEV else (3000, 500, 2000)
NEG_TOP, NEG_RANDOM = 50, 50                 # Unjudged candidates per training query, labeled 0
GBDT_PARAMS = dict(objective="rank:ndcg", eval_metric="ndcg@10", tree_method="hist", max_depth=6,
                   eta=0.1, subsample=0.8, colsample_bytree=0.9, min_child_weight=1, seed=SEED)
GBDT_ROUNDS, GBDT_EARLY_STOP = 300, 30
RUN_BATCH = 16

# ---- OpenSearch ----------------------------------------------------------------------------
OS_VERSION = "2.19.1"
OS_HEAP = "4g"
HNSW_M, HNSW_EF_CONSTRUCTION = 16, 128
OS_FORCE_MERGE = True                        # One segment, one HNSW graph: slow build, fastest queries

# ---- Latency, throughput, cost ------------------------------------------------------------
LAT_QUERIES, LAT_WARMUP = (100, 10) if DEV else (300, 20)
SWEEP_BATCHES, SWEEP_ROUNDS = (1, 4, 8, 16, 32), (5 if DEV else 15)
SLO_P99_MS = {"lexical": 20.0, "pipeline": 100.0}
GPU_HOURLY = 1.76                            # $/hour for the GPU in use (A100 40GB median; L4 about 0.80)
VCPU_HOURLY = 0.0446                         # $/vCPU-hour (c7i.4xlarge on-demand: 0.714 / 16)
CATALOG_VERSION = "v1"                       # Part of the page-cache key

MAX_BATCH = max(EVAL_BATCH, RUN_BATCH, *SWEEP_BATCHES)
RESULTS_FILE = DATA_DIR / ("results_dev.json" if DEV else "results.json")
RESULTS = {}
rng_global = np.random.default_rng(SEED)

## 1 · Data

ESCI US products and the reduced query set (`small_version == 1`), which is the ESCI Task 1 set. The corpus is every product in the locale, so retrieval runs against the full catalog. Training queries are split into **fit** (for tuning and ranker training) and **validation** (for selecting settings); test queries are used only for final numbers.

In [ ]:
BASE_URL = "https://github.com/amazon-science/esci-data/raw/main/shopping_queries_dataset/"
PRODUCTS_FILE = DATA_DIR / "shopping_queries_dataset_products.parquet"
EXAMPLES_FILE = DATA_DIR / "shopping_queries_dataset_examples.parquet"
PRODUCT_COLS = ["product_id"] + list(COLUMN_OF.values())
EXAMPLE_COLS = ["query_id", "query", "product_id", "esci_label", "split"]


def is_parquet(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size < 1024:
        return False
    with open(path, "rb") as f:
        head = f.read(4)
        f.seek(-4, os.SEEK_END)
        return head == b"PAR1" and f.read(4) == b"PAR1"


def load_from_github():
    for path in (PRODUCTS_FILE, EXAMPLES_FILE):
        if not is_parquet(path):
            print("Downloading", path.name)
            tmp = path.with_suffix(".tmp")
            urllib.request.urlretrieve(BASE_URL + path.name, tmp)
            if not is_parquet(tmp):   # A failed Git LFS download leaves a small text file.
                tmp.unlink(missing_ok=True)
                raise IOError(f"{path.name} is not a valid Parquet file")
            tmp.replace(path)
    filt = [("product_locale", "==", LOCALE)]
    products = pd.read_parquet(PRODUCTS_FILE, columns=PRODUCT_COLS, filters=filt)
    examples = pd.read_parquet(EXAMPLES_FILE, columns=EXAMPLE_COLS, filters=filt + [("small_version", "==", 1)])
    return products, examples


def load_from_huggingface():
    from datasets import load_dataset
    columns = sorted(set(PRODUCT_COLS + EXAMPLE_COLS + ["product_locale", "small_version"]) - {"split"})
    parts = []
    for split in ("train", "test"):
        ds = load_dataset("tasksource/esci", split=split).select_columns(columns)
        ds = ds.filter(lambda b: [l == LOCALE for l in b["product_locale"]], batched=True)
        df = ds.to_pandas()
        df["split"] = split
        parts.append(df)
    full = pd.concat(parts, ignore_index=True)
    products = full[PRODUCT_COLS].drop_duplicates("product_id")
    examples = full.loc[full["small_version"].astype(int) == 1, EXAMPLE_COLS]
    return products, examples


try:
    products, examples = load_from_github()
except Exception as err:
    print(f"GitHub load failed ({type(err).__name__}: {err}); trying Hugging Face")
    products, examples = load_from_huggingface()

products = products.drop_duplicates("product_id").reset_index(drop=True)
assert set(examples["esci_label"].unique()) <= set(GAIN), "unexpected ESCI labels"
if MAX_DOCS is not None and MAX_DOCS < len(products):
    judged = products["product_id"].isin(set(examples["product_id"]))
    extra = products[~judged].sample(n=max(0, MAX_DOCS - int(judged.sum())), random_state=SEED)
    products = pd.concat([products[judged], extra]).sort_index().reset_index(drop=True)

doc_of = pd.Series(np.arange(len(products), dtype=np.int64), index=products["product_id"])
examples = examples[examples["product_id"].isin(doc_of.index)].copy()
examples["doc"] = doc_of.loc[examples["product_id"]].to_numpy()
examples["gain"] = examples["esci_label"].map(GAIN).astype(np.float32)
examples["exact"] = examples["esci_label"] == "E"
examples = examples.sample(frac=1.0, random_state=SEED).reset_index(drop=True)   # No tie order from the file
query_text = examples.drop_duplicates("query_id").set_index("query_id")["query"].astype(str)

train_q = rng_global.permutation(examples.loc[examples["split"] == "train", "query_id"].unique())
n_val = int(len(train_q) * VAL_FRACTION)
VAL_IDS, FIT_IDS = train_q[:n_val], train_q[n_val:]
SEL_IDS = VAL_IDS[:N_SELECT]
TEST_IDS = rng_global.permutation(examples.loc[examples["split"] == "test", "query_id"].unique())

print(f"Products: {len(products):,} | judgments: {len(examples):,}")
print(f"Queries: fit {len(FIT_IDS):,} | validation {len(VAL_IDS):,} (selection {len(SEL_IDS):,}) | test {len(TEST_IDS):,}")
print(examples["esci_label"].value_counts(normalize=True).round(3).to_dict())

Products: 1,215,854 | judgments: 601,354
Queries: fit 18,800 | validation 2,088 (selection 1,000) | test 8,956
{'E': 0.435, 'S': 0.351, 'I': 0.169, 'C': 0.045}


## 2 · Text analysis and the GPU BM25F index

The analyzer follows the Lucene `english` analyzer: lowercase, possessive removal, the same 33 stop words, Porter stemming. Postings store one `uint8` term frequency per field, so field weights, per-field `b` and `k1` can change on every request.

**Scoring (BM25F, per-field length normalization):**

$$\widetilde{tf}(t,d)=\sum_f w_f\frac{tf_f(t,d)}{1-b_f+b_f\,|d_f|/\overline{|d_f|}},\qquad s(q,d)=\sum_{t\in q}\mathrm{idf}(t)\,\frac{\widetilde{tf}\,(k_1+1)}{k_1+\widetilde{tf}}$$

`idf` uses document frequency across all fields. When only one field has weight, it uses that field's document frequency, which is exactly what Lucene does for a single-field query, so single-field scores match OpenSearch.

In [4]:
_WORD = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")
EN_STOP = frozenset("a an and are as at be but by for if in into is it no not of on or such "
                    "that the their then there these they this to was will with".split())
_stemmer = snowballstemmer.stemmer("porter")
_STEM_CACHE = {}


def analyze(text):
    out = []
    for w in _WORD.findall(str(text).lower()):
        if w.endswith("'s"):
            w = w[:-2]
        w = w.replace("'", "")
        if not w or w in EN_STOP:
            continue
        s = _STEM_CACHE.get(w)
        if s is None:
            s = _STEM_CACHE[w] = _stemmer.stemWord(w)
        out.append(s)
    return out


def build_postings(field_text):
    # Term-sorted postings with one tf column per field, plus global and per-field idf.
    names = list(field_text)
    N, F = len(field_text[names[0]]), len(names)
    cv = CountVectorizer(analyzer=analyze, dtype=np.int32)
    counts = cv.fit_transform(chain.from_iterable(field_text[f] for f in names)).tocsr()
    V = counts.shape[1]
    keys, values = [], []
    field_len = np.zeros((N, F), dtype=np.float32)
    for f in range(F):
        block = counts[f * N:(f + 1) * N]
        field_len[:, f] = np.asarray(block.sum(axis=1)).ravel()
        coo = block.tocoo()
        keys.append(coo.col.astype(np.int64) * N + coo.row)
        values.append(np.minimum(coo.data, 255).astype(np.uint8))
    del counts
    postings = np.unique(np.concatenate(keys))
    tf = np.zeros((len(postings), F), dtype=np.uint8)
    for f in range(F):
        tf[np.searchsorted(postings, keys[f]), f] = values[f]
    del keys, values
    terms, docs = postings // N, postings % N
    idf_of = lambda df: np.log1p((N - df + 0.5) / (df + 0.5)).astype(np.float32)
    df = np.bincount(terms, minlength=V)
    df_field = np.stack([np.bincount(terms[tf[:, f] > 0], minlength=V) for f in range(F)], axis=1)
    term_ptr = np.zeros(V + 1, dtype=np.int64)
    term_ptr[1:] = np.cumsum(df)
    avg_len = field_len.mean(axis=0)
    rel_len = field_len / np.where(avg_len > 0, avg_len, 1.0)
    return dict(vocab=cv.vocabulary_, term_ptr=term_ptr, doc_ids=docs.astype(np.int32), tf=tf,
                rel_len=rel_len.astype(np.float32), idf=idf_of(df), idf_field=idf_of(df_field), N=N, fields=names)


cache_id = hashlib.md5(json.dumps([LOCALE, MAX_DOCS, FIELDS, len(products), "v3"]).encode()).hexdigest()[:10]
postings_file = DATA_DIR / f"postings_{cache_id}.pkl"
t0 = time.perf_counter()
if postings_file.exists():
    with open(postings_file, "rb") as f:
        post = pickle.load(f)
    print(f"Loaded cached postings in {time.perf_counter() - t0:.0f}s")
else:
    post = build_postings({f: products[COLUMN_OF[f]].fillna("").astype(str).tolist() for f in FIELDS})
    with open(postings_file, "wb") as f:
        pickle.dump(post, f, protocol=4)
    print(f"Built postings in {time.perf_counter() - t0:.0f}s")
gc.collect()
P = len(post["doc_ids"])
print(f"{post['N']:,} docs | {len(post['idf']):,} terms | {P:,} postings | {P / post['N']:.1f} unique terms per doc")

Built postings in 507s
1,215,854 docs | 720,465 terms | 105,666,243 postings | 86.9 unique terms per doc


In [5]:
@dataclass(frozen=True)
class Params:
    w: tuple
    b: tuple
    k1: float

    @staticmethod
    def of(weights, b=BM25_B, k1=BM25_K1):
        return Params(tuple(float(weights.get(f, 0.0)) for f in FIELDS), (float(b),) * len(FIELDS), float(k1))

    @property
    def single_field(self):
        nz = [i for i, w in enumerate(self.w) if w > 0]
        return nz[0] if len(nz) == 1 else None


def bm25f_impact(tf, rel_len, idf, w, b, k1):
    # Shared by the exhaustive scorer and the differentiable tuner.
    norm = (1.0 - b + b * rel_len).clamp_min(1e-6)
    pseudo_tf = (w * tf / norm).sum(dim=-1)
    return idf * pseudo_tf * (k1 + 1.0) / (k1 + pseudo_tf)


class Index:
    def __init__(self, post, device):
        self.device = torch.device(device)
        self.N, self.F, self.vocab = post["N"], len(post["fields"]), post["vocab"]
        t = lambda a: torch.as_tensor(a, device=self.device)
        self.term_ptr, self.doc_ids, self.tf = t(post["term_ptr"]), t(post["doc_ids"]), t(post["tf"])
        self.rel_len, self.idf, self.idf_field = t(post["rel_len"]), t(post["idf"]), t(post["idf_field"])

    def term_ids(self, text):
        return list(dict.fromkeys(self.vocab[t] for t in analyze(text) if t in self.vocab))

    def params_tensors(self, p):
        make = lambda x: torch.tensor(x, device=self.device, dtype=torch.float32)
        return make(p.w), make(p.b), make(p.k1)

    def idf_for(self, p):
        return self.idf if p.single_field is None else self.idf_field[:, p.single_field]

    def _chunks(self, batch_terms):
        q = torch.tensor([i for i, ts in enumerate(batch_terms) for _ in ts], device=self.device, dtype=torch.long)
        t = torch.tensor([x for ts in batch_terms for x in ts], device=self.device, dtype=torch.long)
        if t.numel() == 0:
            return
        start = self.term_ptr[t]
        n = self.term_ptr[t + 1] - start
        chunk = (n.cumsum(0) - 1) // MAX_POSTINGS_PER_CHUNK
        for c in chunk.unique():
            m = chunk == c
            yield q[m], t[m], start[m], n[m]

    def _expand(self, q, t, start, n):
        offsets = n.cumsum(0) - n
        idx = torch.arange(int(n.sum()), device=self.device) + torch.repeat_interleave(start - offsets, n)
        return torch.repeat_interleave(q, n), torch.repeat_interleave(t, n), idx

    @torch.no_grad()
    def score_into(self, buf, batch_terms, p):
        # Exhaustive scoring of every document into a (queries x N) buffer.
        buf.zero_()
        w, b, k1 = self.params_tensors(p)
        idf = self.idf_for(p)
        for chunk in self._chunks(batch_terms):
            q_rep, t_rep, idx = self._expand(*chunk)
            docs = self.doc_ids[idx].long()
            imp = bm25f_impact(self.tf[idx].to(torch.float32), self.rel_len[docs], idf[t_rep], w, b, k1)
            buf.view(-1).index_add_(0, q_rep * self.N + docs, imp)
        return buf


index = Index(post, DEVICE)
del post
gc.collect()

# ---- Check: the scorer against an independent plain-Python BM25F --------------------------
def reference_bm25f(docs_tokens, query_tokens, p):
    N, F = len(docs_tokens), len(FIELDS)
    lens = np.array([[len(d[f]) for f in range(F)] for d in docs_tokens], dtype=float)
    avg = np.where(lens.mean(0) > 0, lens.mean(0), 1.0)
    one = p.single_field
    out = np.zeros(N)
    for t in dict.fromkeys(query_tokens):
        if one is None:
            df = sum(any(t in d[f] for f in range(F)) for d in docs_tokens)
        else:
            df = sum(t in d[one] for d in docs_tokens)
        if df == 0:
            continue
        idf = math.log1p((N - df + 0.5) / (df + 0.5))
        for i, d in enumerate(docs_tokens):
            ptf = sum(p.w[f] * d[f].count(t) / (1 - p.b[f] + p.b[f] * lens[i, f] / avg[f]) for f in range(F))
            out[i] += idf * ptf * (p.k1 + 1) / (p.k1 + ptf) if ptf > 0 else 0.0
    return out


toy = {f: [] for f in FIELDS}
toy_rng = np.random.default_rng(0)
words = "black running shoes nike leather jacket blue cotton shirt women men kids socks wool".split()
for _ in range(60):
    for f in FIELDS:
        toy[f].append(" ".join(toy_rng.choice(words, size=toy_rng.integers(0, 6))))
toy_index = Index(build_postings(toy), DEVICE)
toy_tokens = [[analyze(toy[f][i]) for f in FIELDS] for i in range(60)]
toy_params = [Params.of(HAND_WEIGHTS), Params.of({"title": 1.0}),
              Params((1.0, 0.4, 2.0, 0.1, 0.05), (0.9, 0.2, 0.0, 0.5, 0.75), 0.8)]
buf = torch.empty(1, 60, device=DEVICE)
for p in toy_params:
    for qtext in ("black running shoes", "wool socks kids", "nike"):
        got = toy_index.score_into(buf, [toy_index.term_ids(qtext)], p)[0].cpu().numpy()
        want = reference_bm25f(toy_tokens, analyze(qtext), p)
        assert np.allclose(got, want, rtol=1e-4, atol=1e-5), (p, qtext, np.abs(got - want).max())
del toy_index, buf
print("CHECK PASSED: GPU BM25F matches the reference implementation (multi-field and single-field idf)")

CHECK PASSED: GPU BM25F matches the reference implementation (multi-field and single-field idf)


## 3 · Evaluation

Two protocols, both with ESCI gains (E 1.0, S 0.1, C 0.01, I 0):

- **Retrieve:** search the whole catalog. Unjudged products count as zero gain, which understates every system equally. Metrics: nDCG@10, and Exact recall at 100 and 1000.
- **Rerank:** order each query's judged products only. This is ESCI Task 1, so `rerank_ndcg_full` is comparable with published numbers (random ordering is about 0.74, the paper's cross-encoder 0.83–0.86).

Differences are reported with paired 95% bootstrap intervals over queries.

In [6]:
@dataclass
class QuerySet:
    name: str
    query_ids: np.ndarray
    texts: list
    terms: list
    n_docs: int
    q: np.ndarray            # Query row for each judged pair
    doc: np.ndarray
    gain: np.ndarray
    exact: np.ndarray
    keys: np.ndarray         # Sorted q * n_docs + doc
    key_order: np.ndarray
    ideal_10: np.ndarray
    ideal_full: np.ndarray
    n_exact: np.ndarray

    def __len__(self):
        return len(self.query_ids)


def discount(ranks):
    return 1.0 / np.log2(ranks + 2.0)


def make_query_set(name, frame, idx):
    frame = frame.reset_index(drop=True)
    uniq = frame.drop_duplicates("query_id")
    q_of = pd.Series(np.arange(len(uniq)), index=uniq["query_id"].to_numpy())
    q = q_of.loc[frame["query_id"]].to_numpy()
    doc, gain, exact = frame["doc"].to_numpy(np.int64), frame["gain"].to_numpy(np.float32), frame["exact"].to_numpy()
    Q = len(uniq)
    order = np.lexsort((-gain, q))
    ranks = np.arange(len(q)) - np.searchsorted(q[order], np.arange(Q))[q[order]]
    d = discount(ranks) * gain[order]
    ideal_full = np.bincount(q[order], weights=d, minlength=Q)
    ideal_10 = np.bincount(q[order], weights=np.where(ranks < 10, d, 0.0), minlength=Q)
    keys = q * idx.N + doc
    key_order = np.argsort(keys)
    texts = uniq["query"].astype(str).tolist()
    return QuerySet(name, uniq["query_id"].to_numpy(), texts, [idx.term_ids(t) for t in texts], idx.N,
                    q, doc, gain, exact, keys[key_order], key_order, ideal_10, ideal_full,
                    np.bincount(q, weights=exact, minlength=Q))


def lookup(qs, ids):
    # For a (queries x k) id matrix: whether each id is judged, and its pair index.
    keys = np.arange(len(qs))[:, None] * qs.n_docs + ids
    pos = np.searchsorted(qs.keys, keys).clip(max=len(qs.keys) - 1)
    hit = (qs.keys[pos] == keys) & (ids >= 0)
    return hit, qs.key_order[pos]


def retrieve_metrics(qs, ids):
    hit, pair = lookup(qs, ids)
    gains = np.where(hit, qs.gain[pair], 0.0)
    exact = hit & qs.exact[pair]
    top = min(10, ids.shape[1])
    dcg = (gains[:, :top] * discount(np.arange(top))).sum(axis=1)
    has_gain, has_exact = qs.ideal_10 > 0, qs.n_exact > 0
    return dict(ndcg_10=dcg[has_gain] / qs.ideal_10[has_gain],
                recall_100=exact[has_exact, :100].sum(axis=1) / qs.n_exact[has_exact],
                recall_1000=exact[has_exact, :1000].sum(axis=1) / qs.n_exact[has_exact])


def candidate_recall(qs, ids):
    hit, pair = lookup(qs, ids)
    has_exact = qs.n_exact > 0
    return (hit & qs.exact[pair])[has_exact].sum(axis=1) / qs.n_exact[has_exact]


def rerank_metrics(qs, scores):
    # `scores` is aligned with the judged pairs; ties keep the shuffled pair order.
    order = np.lexsort((-scores, qs.q))
    q_sorted = qs.q[order]
    Q = len(qs)
    ranks = np.arange(len(order)) - np.searchsorted(q_sorted, np.arange(Q))[q_sorted]
    d = discount(ranks) * qs.gain[order]
    dcg_full = np.bincount(q_sorted, weights=d, minlength=Q)
    dcg_10 = np.bincount(q_sorted, weights=np.where(ranks < 10, d, 0.0), minlength=Q)
    valid = qs.ideal_full > 0
    return dict(rerank_ndcg_10=dcg_10[valid] / qs.ideal_10[valid], rerank_ndcg_full=dcg_full[valid] / qs.ideal_full[valid])


def bootstrap_delta(a, b, n=2000):
    r = np.random.default_rng(SEED)
    delta = a - b
    means = np.empty(n)
    for i in range(0, n, 200):
        draws = r.integers(0, len(delta), size=(min(200, n - i), len(delta)))
        means[i:i + 200] = delta[draws].mean(axis=1)
    return delta.mean(), *np.percentile(means, [2.5, 97.5])


def compare(results, metric, baseline):
    rows = {}
    for name, res in results.items():
        if metric not in res:
            continue
        delta, low, high = bootstrap_delta(res[metric], results[baseline][metric])
        rows[name] = {metric: res[metric].mean(), f"Δ vs {baseline}": delta, "95% low": low, "95% high": high}
    return pd.DataFrame(rows).T


def frame_for(ids):
    return examples[examples["query_id"].isin(set(ids))]


# ---- Check: metrics against hand-computed values ---------------------------------------------
_toy = pd.DataFrame(dict(query_id=[7, 7, 7], query=["q"] * 3, doc=[1, 2, 3], gain=[1.0, 0.1, 0.0], exact=[True, False, False]))
_qs = make_query_set("toy", _toy, type("I", (), {"N": 10, "term_ids": lambda self, t: []})())
_m = retrieve_metrics(_qs, np.array([[2, 5, 1]]))
_want = (0.1 / 1 + 1.0 / math.log2(4)) / (1.0 + 0.1 / math.log2(3))
assert abs(_m["ndcg_10"][0] - _want) < 1e-9 and _m["recall_100"][0] == 1.0
_r = rerank_metrics(_qs, np.array([0.5, 0.9, 0.1]))   # Order: doc 2, doc 1, doc 3
assert abs(_r["rerank_ndcg_full"][0] - (0.1 + 1.0 / math.log2(3)) / (1.0 + 0.1 / math.log2(3))) < 1e-9
assert candidate_recall(_qs, np.array([[3, 2]]))[0] == 0.0
print("CHECK PASSED: metrics match hand-computed values")

FIT = make_query_set("fit", frame_for(FIT_IDS), index)
VAL = make_query_set("val", frame_for(VAL_IDS), index)
SEL = make_query_set("select", frame_for(SEL_IDS), index)
TEST = make_query_set("test", frame_for(TEST_IDS), index)
assert not (set(FIT.query_ids) & set(VAL.query_ids)) and not (set(TEST.query_ids) & (set(FIT.query_ids) | set(VAL.query_ids)))
print("CHECK PASSED: fit, validation and test queries are disjoint")
print(f"Query sets: fit {len(FIT):,} | validation {len(VAL):,} | selection {len(SEL):,} | test {len(TEST):,}")


@torch.no_grad()
def gpu_search(terms, p, k, batch=EVAL_BATCH):
    out, buf = [], None
    for s in range(0, len(terms), batch):
        bt = terms[s:s + batch]
        if buf is None or buf.shape[0] != len(bt):
            buf = torch.empty(len(bt), index.N, device=DEVICE)
        index.score_into(buf, bt, p)
        v, i = buf.topk(min(k, index.N), dim=1)
        out.append(i.masked_fill(v <= 0, -1).cpu().numpy())
    return np.concatenate(out)


@torch.no_grad()
def gpu_rerank_scores(qs, p, batch=EVAL_BATCH):
    scores = np.zeros(len(qs.q), np.float32)
    order = np.argsort(qs.q, kind="stable")
    bounds = np.searchsorted(qs.q[order], np.arange(len(qs) + 1))
    buf = None
    for s in range(0, len(qs), batch):
        e = min(s + batch, len(qs))
        if buf is None or buf.shape[0] != e - s:
            buf = torch.empty(e - s, index.N, device=DEVICE)
        index.score_into(buf, qs.terms[s:e], p)
        pairs = order[bounds[s]:bounds[e]]
        rows = torch.as_tensor(qs.q[pairs] - s, device=DEVICE)
        cols = torch.as_tensor(qs.doc[pairs], device=DEVICE)
        scores[pairs] = buf[rows, cols].cpu().numpy()
    return scores


def evaluate_gpu(qs, p):
    return {**retrieve_metrics(qs, gpu_search(qs.terms, p, EVAL_DEPTH)), **rerank_metrics(qs, gpu_rerank_scores(qs, p))}

CHECK PASSED: metrics match hand-computed values
CHECK PASSED: fit, validation and test queries are disjoint
Query sets: fit 18,800 | validation 2,088 | selection 1,000 | test 8,956


## 4 · Experiment 1a · Tune the GPU BM25F

Gradient tuning of the field weights `w`, per-field `b` and `k1` on the **fit** queries' judged products, with a listwise (ListNet) loss. `w_title` stays at 1, because `w` and `k1` could otherwise scale together without changing the ranking.

Checkpoints are **selected by whole-catalog nDCG@10 on the selection queries**. OpenSearch's boosts are selected on the same queries with the same metric in experiment 1b.

In [7]:
@dataclass
class PairData:
    entry_pair: torch.Tensor     # Judged pair of each (pair, query term) posting
    entry_tf: torch.Tensor       # Per-field tf of that posting
    entry_idf: torch.Tensor
    pair_rel_len: torch.Tensor   # Per-field relative length of each pair's document
    q: torch.Tensor              # Query row of each pair
    target: torch.Tensor         # Gain normalized within the query
    n_pairs: int
    n_queries: int


@torch.no_grad()
def extract_pairs(qs, batch=128):
    lookup = torch.full((batch, index.N), -1, dtype=torch.int32, device=DEVICE)
    pair_ids = np.argsort(qs.q, kind="stable")
    bounds = np.searchsorted(qs.q[pair_ids], np.arange(len(qs) + 1))
    pairs, tfs, idfs = [], [], []
    for s in range(0, len(qs), batch):
        e = min(s + batch, len(qs))
        chosen = pair_ids[bounds[s]:bounds[e]]
        rows = torch.as_tensor(qs.q[chosen] - s, device=DEVICE, dtype=torch.long)
        cols = torch.as_tensor(qs.doc[chosen], device=DEVICE, dtype=torch.long)
        lookup[rows, cols] = torch.as_tensor(chosen, device=DEVICE, dtype=torch.int32)
        for chunk in index._chunks(qs.terms[s:e]):
            q_rep, t_rep, post_idx = index._expand(*chunk)
            hit = lookup[q_rep, index.doc_ids[post_idx].long()]
            keep = hit >= 0
            pairs.append(hit[keep].long())
            tfs.append(index.tf[post_idx[keep]].to(torch.float32))
            idfs.append(index.idf[t_rep[keep]])
        lookup[rows, cols] = -1
    del lookup
    q = torch.as_tensor(qs.q, device=DEVICE, dtype=torch.long)
    gain = torch.as_tensor(qs.gain, device=DEVICE)
    total = torch.zeros(len(qs), device=DEVICE).index_add(0, q, gain)
    return PairData(torch.cat(pairs), torch.cat(tfs), torch.cat(idfs),
                    index.rel_len[torch.as_tensor(qs.doc, device=DEVICE, dtype=torch.long)],
                    q, gain / total[q].clamp_min(1e-12), len(qs.q), len(qs))


def pair_scores(pd_, w, b, k1):
    imp = bm25f_impact(pd_.entry_tf, pd_.pair_rel_len[pd_.entry_pair], pd_.entry_idf, w, b, k1)
    return torch.zeros(pd_.n_pairs, device=DEVICE).index_add(0, pd_.entry_pair, imp)


def listwise_loss(scores, pd_):
    # ListNet: cross-entropy between each query's softmax over its judged products and its normalized gains.
    m = torch.full((pd_.n_queries,), -1e30, device=DEVICE).scatter_reduce(0, pd_.q, scores, "amax")
    lse = torch.log(torch.zeros(pd_.n_queries, device=DEVICE).index_add(0, pd_.q, torch.exp(scores - m[pd_.q]))) + m
    return -(pd_.target * (scores - lse[pd_.q])).sum() / pd_.n_queries


def select_score_gpu(p):
    return retrieve_metrics(SEL, gpu_search(SEL.terms, p, 10))["ndcg_10"].mean()


t0 = time.perf_counter()
FIT_P = extract_pairs(FIT)
print(f"Extracted {FIT_P.entry_pair.numel():,} postings for {FIT_P.n_pairs:,} judged pairs in {time.perf_counter() - t0:.0f}s")

HAND = Params.of(HAND_WEIGHTS)
mask = torch.tensor([0.0 if f == "title" else 1.0 for f in FIELDS], device=DEVICE)
log_w = torch.log(torch.tensor(HAND.w, device=DEVICE) / HAND.w[0]).requires_grad_(True)
logit_b = torch.logit(torch.tensor(HAND.b, device=DEVICE)).requires_grad_(True)
log_k1 = torch.log(torch.tensor(HAND.k1, device=DEVICE)).requires_grad_(True)
opt = torch.optim.Adam([log_w, logit_b, log_k1], lr=GRAD_LR)
to_params = lambda: (torch.exp(log_w * mask), torch.sigmoid(logit_b), torch.exp(log_k1))

history, best_p, best_val = [], HAND, select_score_gpu(HAND)
t0 = time.perf_counter()
for step in range(GRAD_STEPS + 1):
    w, b, k1 = to_params()
    loss = listwise_loss(pair_scores(FIT_P, w, b, k1), FIT_P)
    if step % SELECT_EVERY == 0 or step == GRAD_STEPS:
        p = Params(tuple(w.tolist()), tuple(b.tolist()), float(k1))
        val = select_score_gpu(p)
        history.append(dict(step=step, loss=float(loss), select_ndcg_10=val))
        if val > best_val:
            best_p, best_val = p, val
    if step < GRAD_STEPS:
        opt.zero_grad()
        loss.backward()
        opt.step()
BM25F_TUNED = best_p
del FIT_P
gc.collect()
print(f"Tuned in {time.perf_counter() - t0:.0f}s. Selection nDCG@10: hand {select_score_gpu(HAND):.4f} -> tuned {best_val:.4f}")
display(pd.DataFrame(history).iloc[::max(1, len(history) // 8)])
display(pd.DataFrame({"hand w": HAND.w, "tuned w": BM25F_TUNED.w, "hand b": HAND.b, "tuned b": BM25F_TUNED.b},
                     index=FIELDS).round(3))
print(f"k1: hand {HAND.k1:.3f} -> tuned {BM25F_TUNED.k1:.3f}")

Extracted 839,593 postings for 377,860 judged pairs in 1s


/tmp/ipykernel_1287/2477523161.py:76: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  p = Params(tuple(w.tolist()), tuple(b.tolist()), float(k1))


Tuned in 6s. Selection nDCG@10: hand 0.3022 -> tuned 0.3120


,step,loss,select_ndcg_10
0,0,7.0040,0.2760
3,30,5.7174,0.2955
6,60,5.5393,0.3025
9,90,5.4835,0.3045
12,120,5.4601,0.3066
15,150,5.4473,0.3070
18,180,5.4394,0.3088
21,210,5.4341,0.3105
24,240,5.4304,0.3103
27,270,5.4277,0.3103


,hand w,tuned w,hand b,tuned b
title,3.0000,1.0000,0.7500,0.7420
brand,2.0000,0.1260,0.7500,0.6750
color,1.0000,0.1400,0.7500,0.4750
bullet_point,1.0000,0.0600,0.7500,0.6190
description,0.5000,0.0360,0.7500,0.5510


k1: hand 1.200 -> tuned 0.036


## 5 · Shared assets: facets, embeddings, cuVS, OpenSearch index

One OpenSearch index holds everything both experiments need, built from the same products as the GPU index, with OpenSearch `_id` equal to the GPU row number:

- the five text fields with the `english` analyzer, for `match` and `cross_fields`;
- `weighted_all`: each field's text repeated in proportion to the GPU-tuned weights (at most `WCONCAT_MAX_REPS` times), with BM25 `k1` set to the tuned value. This is the closest stock OpenSearch gets to BM25F; it has one `b` for the combined field;
- `brand_kw` and `color_kw` keyword fields for facets and filters;
- `vec`: the product embeddings in a faiss HNSW graph.

In [8]:
# ---- Facet columns, static features, display fields: shared by both engines ----------------
TITLES = products["product_title"].fillna("").astype(str).to_numpy()


def facet_column(series):
    # Same rule as the OpenSearch keyword field: trimmed, values over the length limit dropped.
    vals = series.fillna("").astype(str).str.strip()
    vals = vals.where(vals.str.len() <= KEYWORD_MAX_CHARS, "")
    names = pd.Index([""]).append(pd.Index(vals[vals != ""].unique()))
    return vals.to_numpy(dtype=object), names.to_numpy(dtype=object), names.get_indexer(vals).astype(np.int64)


FACET = {}
for f in FACET_FIELDS:
    values, names, codes = facet_column(products[COLUMN_OF[f]])
    toks = [frozenset(analyze(n)) for n in names]
    inv = collections.defaultdict(list)
    for c, ts in enumerate(toks):
        for t in ts:
            inv[t].append(c)
    FACET[f] = dict(values=values, names=names, codes=codes, n=len(names), toks=toks, inv=inv,
                    code_of={n: i for i, n in enumerate(names)}, os_field=f"{f}_kw",
                    codes_t=torch.as_tensor(codes, device=DEVICE), present_t=torch.as_tensor(codes > 0, device=DEVICE))
    print(f"{f}: {len(names) - 1:,} distinct values; {np.mean(codes == 0):.1%} of products have none")

brand_count = np.bincount(FACET["brand"]["codes"], minlength=FACET["brand"]["n"]).astype(np.float32)
brand_count[0] = 0
STATIC_NAMES = [f"len_{f}" for f in FIELDS] + ["brand_pop_log"]
STATIC = np.column_stack([index.rel_len.float().cpu().numpy(),
                          np.log1p(brand_count[FACET["brand"]["codes"]])]).astype(np.float32)
STATIC_T = torch.as_tensor(STATIC, device=DEVICE)
FEATURES = (["lex_score", "lex_norm", "lex_rank_log", "dense_sim", "dense_gap", "dense_rank_log",
             "rrf", "in_both", "brand_match", "color_match"] + STATIC_NAMES + ["q_len", "n_match_log"])


def matched_codes(q_tokens, f):
    # Facet values whose analyzed tokens all appear in the query, e.g. brand "levi's" in "levis 501 jeans".
    fc = FACET[f]
    cands = {c for t in q_tokens for c in fc["inv"].get(t, ())}
    return np.array(sorted(c for c in cands if fc["toks"][c] <= q_tokens), dtype=np.int64)


def query_info(texts, need_terms):
    out = []
    for t in texts:
        toks = analyze(t)
        q = set(toks)
        out.append(dict(text=t, n_tok=len(toks), brand=matched_codes(q, "brand"), color=matched_codes(q, "color"),
                        terms=index.term_ids(t) if need_terms else None))
    return out

brand: 211,156 distinct values; 5.9% of products have none
color: 175,292 distinct values; 33.4% of products have none


In [9]:
# ---- Embeddings: products once (cached), queries online -----------------------------------------
from sentence_transformers import SentenceTransformer

emb_file = DATA_DIR / f"emb_{re.sub(r'[^A-Za-z0-9]+', '_', EMBED_MODEL)}_{cache_id}.npy"
encoder = SentenceTransformer(EMBED_MODEL, device=str(DEVICE))
encoder.max_seq_length = EMBED_MAX_LEN
if IS_CUDA:
    encoder.half()

if emb_file.exists():
    EMB_NP = np.load(emb_file)
    print("Loaded product embeddings", EMB_NP.shape)
else:
    product_text = (products["product_title"].fillna("").astype(str) + " | "
                    + FACET["brand"]["values"] + " | " + FACET["color"]["values"]).tolist()
    t0 = time.perf_counter()
    EMB_NP = encoder.encode(product_text, batch_size=EMBED_BATCH, normalize_embeddings=True,
                            convert_to_numpy=True, show_progress_bar=True).astype(np.float16)
    np.save(emb_file, EMB_NP)
    print(f"Embedded {len(product_text):,} products in {time.perf_counter() - t0:.0f}s")
    del product_text

EMB = torch.as_tensor(EMB_NP, device=DEVICE)
EMB = EMB if IS_CUDA else EMB.float()
DIM = EMB.shape[1]


@torch.no_grad()
def encode_queries(texts, batch_size=512):
    v = encoder.encode([QUERY_PREFIX + t for t in texts], batch_size=batch_size, normalize_embeddings=True,
                       convert_to_tensor=True, show_progress_bar=False)
    return v.to(device=DEVICE, dtype=EMB.dtype)


print(f"Embeddings: {EMB.shape[0]:,} x {DIM}, {EMB.element_size() * EMB.numel() / 2**30:.2f} GB on {EMB.device}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1188 [00:00<?, ?it/s]

Embedded 1,215,854 products in 153s
Embeddings: 1,215,854 x 384, 0.87 GB on cuda:0


In [10]:
# ---- GPU dense retrieval with cuVS ----------------------------------------------------------
def pack_bitmap(mask):
    # Row-major bits, least significant bit first, as uint32 words.
    flat = mask.reshape(-1)
    pad = (-flat.numel()) % 32
    if pad:
        flat = torch.cat([flat, flat.new_zeros(pad)])
    words = (flat.view(-1, 32).to(torch.int64) << torch.arange(32, device=flat.device)).sum(1)
    words = torch.where(words >= 2**31, words - 2**32, words).to(torch.int32)
    return cp.from_dlpack(words).view(cp.uint32)


def row_overlap(a, b, n):
    a, b = a[:, :n].cpu().numpy(), b[:, :n].cpu().numpy()
    return float(np.mean([len(set(x[x >= 0]) & set(y[y >= 0])) / max(1, len(set(y[y >= 0]))) for x, y in zip(a, b)]))


class DenseIndex:
    # Top-k inner product. Similarities of returned ids are recomputed exactly, so every backend
    # hands the ranker the same values. Filtered search checks cuVS's bitmap convention once.

    def __init__(self, emb, backend):
        self.emb, self.used, self.prefilter = emb, "torch", None
        if not (IS_CUDA and backend.startswith("cuvs")):
            return
        try:
            from cuvs.neighbors import brute_force, cagra, filters
            self.bf, self.cagra, self.filters = brute_force, cagra, filters
            self.data = cp.from_dlpack(emb)
            t0 = time.perf_counter()
            self.bf_index = brute_force.build(self.data, metric="sqeuclidean")   # Same order as cosine on unit vectors
            if backend == "cuvs_cagra":
                self.cg_index = cagra.build(cagra.IndexParams(metric="sqeuclidean", graph_degree=64,
                                                              intermediate_graph_degree=128), self.data)
            torch.cuda.synchronize()
            self.used = backend
            print(f"cuVS {backend} built in {time.perf_counter() - t0:.1f}s")
        except Exception as err:
            print(f"cuVS unavailable ({type(err).__name__}: {err}); using exact torch search")

    def exact(self, q, k, filt=None):
        sims = (q @ self.emb.T).float()
        if filt is not None:
            sims.masked_fill_(~filt, float("-inf"))
        v, i = sims.topk(min(k, sims.shape[1]), dim=1)
        return i.masked_fill(torch.isinf(v), -1), v

    def _finish(self, q, ids):
        ids = ids.long()
        ids = ids.masked_fill((ids < 0) | (ids >= self.emb.shape[0]), -1)
        sims = torch.bmm(self.emb[ids.clamp_min(0)], q.unsqueeze(2)).squeeze(2).float()
        sims.masked_fill_(ids < 0, float("-inf"))
        sims, order = sims.sort(dim=1, descending=True)
        return ids.gather(1, order), sims

    def _cuvs(self, q, k, filt):
        qc = cp.from_dlpack(q.contiguous())
        if filt is None and self.used == "cuvs_cagra":
            try:
                _, nb = self.cagra.search(self.cagra.SearchParams(itopk_size=max(256, k)), self.cg_index, qc, k)
            except Exception as err:
                print(f"CAGRA failed for k={k} ({type(err).__name__}: {err}); switching to brute force")
                self.used = "cuvs_brute_force"
                return self._cuvs(q, k, filt)
        elif filt is None:
            _, nb = self.bf.search(self.bf_index, qc, k)
        else:
            bits = pack_bitmap(filt if self.prefilter == "plain" else ~filt)
            _, nb = self.bf.search(self.bf_index, qc, k, prefilter=self.filters.from_bitmap(bits))
        return self._finish(q, torch.as_tensor(cp.asarray(nb).astype(cp.int64), device=q.device))

    def _calibrate(self, q, k, filt):
        ref, _ = self.exact(q, k, filt)
        for mode in ("plain", "inverted"):
            self.prefilter = mode
            try:
                ids, _ = self._cuvs(q, k, filt)
            except Exception as err:
                print(f"cuVS prefilter failed ({type(err).__name__}: {err})")
                break
            ok = ids >= 0
            if bool(filt.gather(1, ids.clamp_min(0))[ok].all()) and row_overlap(ids, ref, min(100, k)) > 0.99:
                print(f"cuVS prefiltered search enabled ({mode} bitmap)")
                return
        self.prefilter = False
        print("cuVS prefilter not usable; filtered dense search uses exact torch search")

    @torch.no_grad()
    def search(self, q, k, filt=None):
        k = min(k, self.emb.shape[0])
        if self.used == "torch":
            return self.exact(q, k, filt)
        if filt is not None and self.prefilter is None:
            self._calibrate(q, k, filt)
        if filt is not None and self.prefilter is False:
            return self.exact(q, k, filt)
        return self._cuvs(q, k, filt)


DENSE = DenseIndex(EMB, DENSE_BACKEND)
probe = encode_queries(TEST.texts[:32])
ids_a, _ = DENSE.search(probe, K_DENSE)
ids_b, _ = DENSE.exact(probe, K_DENSE)
ov = row_overlap(ids_a, ids_b, K_DENSE)
print(f"Dense backend {DENSE.used}: overlap with exact search @100 {row_overlap(ids_a, ids_b, 100):.3f}, @{K_DENSE} {ov:.3f}")
if DENSE.used == "cuvs_brute_force":
    assert ov > 0.99, "brute-force cuVS should match exact search"
    print("CHECK PASSED: cuVS brute force matches exact search")

cuVS unavailable (ImportError: libcuvs_c.so: cannot open shared object file: No such file or directory); using exact torch search
Dense backend torch: overlap with exact search @100 1.000, @5000 1.000


In [13]:

!grep -E "OutOfMemory|FATAL|ERROR|Exception|killed" /content/opensearch.log | tail -n 20
!tail -n 40 /content/opensearch.log
!ps aux | grep -c "[o]rg.opensearch.bootstrap"
!dmesg 2>/dev/null | grep -iE "killed process|out of memory" | tail -n 5
!free -g; df -h /content

[2026-11-04T18:35:36,834][INFO ][o.o.n.Node               ] [c7d45c4972bf] JVM arguments [-Xshare:auto, -Dopensearch.networkaddress.cache.ttl=60, -Dopensearch.networkaddress.cache.negative.ttl=10, -XX:+AlwaysPreTouch, -Xss1m, -Djava.awt.headless=true, -Dfile.encoding=UTF-8, -Djna.nosys=true, -XX:-OmitStackTraceInFastThrow, -XX:+ShowCodeDetailsInExceptionMessages, -Dio.netty.noUnsafe=true, -Dio.netty.noKeySetOptimization=true, -Dio.netty.recycler.maxCapacityPerThread=0, -Dio.netty.allocator.numDirectArenas=0, -Dlog4j.shutdownHookEnabled=false, -Dlog4j2.disable.jmx=true, -Djava.security.manager=allow, -Djava.locale.providers=SPI,COMPAT, -Xms1g, -Xmx1g, -XX:+UseG1GC, -XX:G1ReservePercent=25, -XX:InitiatingHeapOccupancyPercent=30, -Djava.io.tmpdir=/tmp/opensearch-10755457180676600529, -XX:+HeapDumpOnOutOfMemoryError, -XX:HeapDumpPath=data, -XX:ErrorFile=logs/hs_err_pid%p.log, -Xlog:gc*,gc+age=trace,safepoint:file=logs/gc.log:utctime,pid,tags:filecount=32,filesize=64m, -Djava.security.manag

In [12]:
# ---- Start OpenSearch and build its index -------------------------------------------------------
from opensearchpy import OpenSearch, helpers

OS_HOME = Path(f"/content/opensearch-{OS_VERSION}")
OS_URL = f"https://artifacts.opensearch.org/releases/bundle/opensearch/{OS_VERSION}/opensearch-{OS_VERSION}-linux-x64.tar.gz"
client = OpenSearch("http://localhost:9200", timeout=600, pool_maxsize=16)


def os_alive():
    try:
        return client.ping()
    except Exception:
        return False


def start_opensearch():
    if os_alive():
        print("OpenSearch already running")
        return
    if not (OS_HOME / "bin" / "opensearch").exists():
        tgz = Path(f"/content/opensearch-{OS_VERSION}.tar.gz")
        if not tgz.exists():
            print("Downloading OpenSearch", OS_VERSION)
            urllib.request.urlretrieve(OS_URL, tgz)
        with tarfile.open(tgz) as t:
            t.extractall("/content")
    subprocess.run(["useradd", "-m", "osuser"], stderr=subprocess.DEVNULL)   # OpenSearch refuses to run as root
    subprocess.run(["chown", "-R", "osuser:osuser", str(OS_HOME)], check=True)
    cmd = ["runuser", "-u", "osuser", "--", "env", f"OPENSEARCH_JAVA_OPTS=-Xms{OS_HEAP} -Xmx{OS_HEAP}",
           str(OS_HOME / "bin" / "opensearch"), "-Ediscovery.type=single-node",
           "-Eplugins.security.disabled=true", "-Ehttp.port=9200"]
    log = Path("/content/opensearch.log")
    subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(90):
        if os_alive():
            print("OpenSearch is up:", client.info()["version"]["number"])
            return
        time.sleep(2)
    raise RuntimeError(f"OpenSearch did not start:\n{log.read_text()[-3000:]}")


start_opensearch()

max_w = max(BM25F_TUNED.w)
REPS = {f: max(1, round(WCONCAT_MAX_REPS * w / max_w)) for f, w in zip(FIELDS, BM25F_TUNED.w)}
OS_INDEX = "esci_" + hashlib.md5(json.dumps([cache_id, REPS, round(BM25F_TUNED.k1, 3), EMBED_MODEL, DIM, HNSW_M,
                                             HNSW_EF_CONSTRUCTION, KEYWORD_MAX_CHARS]).encode()).hexdigest()[:10]
print("Weighted concatenation repeats:", REPS, "| index:", OS_INDEX)


def clean_text(v):
    return "" if v is None or (isinstance(v, float) and math.isnan(v)) else str(v)


def os_index_body(with_ef_search=True):
    hnsw = {"m": HNSW_M, "ef_construction": HNSW_EF_CONSTRUCTION}
    if with_ef_search:
        hnsw["ef_search"] = K_DENSE
    text = {"type": "text", "analyzer": "english"}
    return {
        "settings": {"index": {
            "number_of_shards": 1, "number_of_replicas": 0, "refresh_interval": "-1", "knn": True,
            "max_result_window": max(K_LEX, K_DENSE, EVAL_DEPTH) + 1000,
            "similarity": {"wconcat": {"type": "BM25", "k1": round(BM25F_TUNED.k1, 4), "b": BM25_B}}}},
        "mappings": {"_source": {"enabled": False}, "properties": {
            **{f: dict(text) for f in FIELDS},
            "weighted_all": {**text, "similarity": "wconcat"},
            **{FACET[f]["os_field"]: {"type": "keyword", "ignore_above": KEYWORD_MAX_CHARS} for f in FACET_FIELDS},
            "vec": {"type": "knn_vector", "dimension": DIM, "method": {
                "name": "hnsw", "engine": "faiss", "space_type": "innerproduct", "parameters": hnsw}},
        }},
    }


def os_actions():
    cols = [COLUMN_OF[f] for f in FIELDS]
    for start in range(0, index.N, 10_000):
        chunk = products.iloc[start:start + 10_000][cols]
        vecs = np.round(EMB_NP[start:start + len(chunk)].astype(np.float32), 5)
        for i, row in enumerate(chunk.itertuples(index=False, name=None)):
            doc = start + i
            src = dict(zip(FIELDS, map(clean_text, row)))
            src["weighted_all"] = " ".join(" ".join([src[f]] * REPS[f]) for f in FIELDS if src[f])
            for f in FACET_FIELDS:
                if FACET[f]["values"][doc]:
                    src[FACET[f]["os_field"]] = FACET[f]["values"][doc]
            src["vec"] = vecs[i].tolist()
            yield {"_index": OS_INDEX, "_id": str(doc), "_source": src}


def build_os_index():
    if client.indices.exists(index=OS_INDEX):
        client.indices.refresh(index=OS_INDEX)
        if client.count(index=OS_INDEX)["count"] == index.N:
            print(f"Reusing {OS_INDEX}")
            return
        client.indices.delete(index=OS_INDEX)
    client.cluster.put_settings(body={"persistent": {"knn.algo_param.index_thread_qty": max(1, (os.cpu_count() or 2) // 2)}})
    try:
        client.indices.create(index=OS_INDEX, body=os_index_body())
    except Exception as err:
        if "ef_search" not in str(err):
            raise
        print("Mapping rejected ef_search; queries pass it as method_parameters instead")
        client.indices.create(index=OS_INDEX, body=os_index_body(with_ef_search=False))
    t0, failed = time.perf_counter(), 0
    for i, (ok, _) in enumerate(helpers.parallel_bulk(client, os_actions(), thread_count=4, chunk_size=500,
                                                      max_chunk_bytes=50 * 2**20, request_timeout=600)):
        failed += not ok
        if i and i % 200_000 == 0:
            print(f"  indexed {i:,} ({time.perf_counter() - t0:.0f}s)")
    client.indices.put_settings(index=OS_INDEX, body={"index": {"refresh_interval": None}})
    client.indices.refresh(index=OS_INDEX, request_timeout=3600)
    print(f"Bulk indexing: {time.perf_counter() - t0:.0f}s, {failed} failures")
    if OS_FORCE_MERGE:
        t1 = time.perf_counter()
        client.indices.forcemerge(index=OS_INDEX, max_num_segments=1, request_timeout=7200)
        print(f"Force merge (rebuilds the HNSW graph): {time.perf_counter() - t1:.0f}s")
    n = client.count(index=OS_INDEX)["count"]
    assert n == index.N and failed == 0, f"indexed {n:,} of {index.N:,}"


build_os_index()
try:
    client.transport.perform_request("GET", f"/_plugins/_knn/warmup/{OS_INDEX}")
except Exception as err:
    print(f"k-NN warmup skipped ({type(err).__name__})")

OS_KNN_METHOD_PARAMS = True
OS_FILTER_PATH = ["took", "responses.took", "responses.error", "responses.hits.total.value",
                  "responses.hits.hits._id", "responses.hits.hits._score",
                  "responses.aggregations.*.buckets.key", "responses.aggregations.*.buckets.doc_count"]


def os_knn(vec, k, flt=None):
    clause = {"vector": vec, "k": k}
    if OS_KNN_METHOD_PARAMS:
        clause["method_parameters"] = {"ef_search": max(k, 100)}
    if flt is not None:
        clause["filter"] = flt
    return {"knn": {"vec": clause}}


def os_ip_from_score(s):
    # faiss innerproduct _score: ip + 1 when ip >= 0, else 1 / (1 - ip).
    s = np.asarray(s, dtype=np.float64)
    return np.where(s >= 1.0, s - 1.0, 1.0 - 1.0 / np.maximum(s, 1e-12))


def os_msearch(bodies):
    payload = []
    for b in bodies:
        payload += [{"index": OS_INDEX}, b]
    resp = client.msearch(body=payload, filter_path=OS_FILTER_PATH, request_timeout=600)
    for r in resp["responses"]:
        if "error" in r:
            raise RuntimeError(str(r["error"])[:500])
    return resp


def parse_hits(r):
    hits = r.get("hits", {}).get("hits", [])
    ids = np.fromiter((int(h["_id"]) for h in hits), dtype=np.int64, count=len(hits))
    return ids, np.fromiter((h["_score"] for h in hits), dtype=np.float64, count=len(hits))


# ---- Check: k-NN options and the score conversion, against exact similarities -----------------------
pv = probe[:1].float().cpu().numpy()[0]
try:
    r = client.search(index=OS_INDEX, body={"query": os_knn(pv.tolist(), 100), "size": 100, "_source": False})
except Exception as err:
    if "method_parameters" not in str(err):
        raise
    OS_KNN_METHOD_PARAMS = False
    print("This version rejects method_parameters; using the mapping's ef_search")
    r = client.search(index=OS_INDEX, body={"query": os_knn(pv.tolist(), 100), "size": 100, "_source": False})
hit_ids, hit_sc = parse_hits(r)
err = np.abs(os_ip_from_score(hit_sc) - EMB_NP[hit_ids].astype(np.float32) @ pv).max()
ref_ids, _ = DENSE.exact(probe[:1], 100)
assert err < 0.01, f"k-NN score conversion is off by {err:.4f}"
print(f"CHECK PASSED: OpenSearch k-NN scores convert to inner products (max error {err:.4f}); "
      f"HNSW overlap with exact top 100: {len(set(hit_ids) & set(ref_ids[0].cpu().numpy())) / 100:.2f}")

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/opensearchpy/connection/http_urllib3.py", line 280, in perform_request
    response = self.pool.urlopen(
        method, url, body, retries=Retry(False), headers=request_headers, **kw
    )
  File "/usr/local/lib/python3.13/dist-packages/urlli

OpenSearch is up: 2.19.1
Weighted concatenation repeats: {'title': 8, 'brand': 1, 'color': 1, 'bullet_point': 1, 'description': 1} | index: esci_3e89c3ae43


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
        conn,
    ...<10 lines>...
        **response_kw,
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
  File "/usr/lib/python3.13/http/client.py", line 1478, in getresponse
    response.begin()
    ~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/http/client.py", line 343, in begin
    version, status, reason = self._read_status()
                              ~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/http/client.py", line 312, in _read_status
    raise RemoteDisconnected("Remote end closed connection without"
                             " response")
http.client.RemoteDisconnected: 

ConnectionError: ConnectionError(<urllib3.connection.HTTPConnection object at 0x7973c9f9bac0>: Failed to establish a new connection: [Errno 111] Connection refused) caused by: NewConnectionError(<urllib3.connection.HTTPConnection object at 0x7973c9f9bac0>: Failed to establish a new connection: [Errno 111] Connection refused)

In [15]:

# ---- Start OpenSearch and build its index -------------------------------------------------------
from opensearchpy import OpenSearch, helpers

OS_HOME = Path(f"/content/opensearch-{OS_VERSION}")
OS_URL = f"https://artifacts.opensearch.org/releases/bundle/opensearch/{OS_VERSION}/opensearch-{OS_VERSION}-linux-x64.tar.gz"
client = OpenSearch("http://localhost:9200", timeout=600, pool_maxsize=16)


def os_alive():
    try:
        return client.ping()
    except Exception:
        return False


def start_opensearch():
    if os_alive():
        print("OpenSearch already running")
        return
    if not (OS_HOME / "bin" / "opensearch").exists():
        tgz = Path(f"/content/opensearch-{OS_VERSION}.tar.gz")
        if not tgz.exists():
            print("Downloading OpenSearch", OS_VERSION)
            urllib.request.urlretrieve(OS_URL, tgz)
        with tarfile.open(tgz) as t:
            t.extractall("/content")
    subprocess.run(["useradd", "-m", "osuser"], stderr=subprocess.DEVNULL)   # OpenSearch refuses to run as root
    subprocess.run(["chown", "-R", "osuser:osuser", str(OS_HOME)], check=True)
    knn_lib = OS_HOME / "plugins" / "opensearch-knn" / "lib"   # faiss JNI libraries and their dependencies
    assert knn_lib.exists(), f"k-NN native libraries not found in {knn_lib}"
    cmd = ["runuser", "-u", "osuser", "--", "env", f"OPENSEARCH_JAVA_OPTS=-Xms{OS_HEAP} -Xmx{OS_HEAP}",
           f"LD_LIBRARY_PATH={knn_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}",
           str(OS_HOME / "bin" / "opensearch"), "-Ediscovery.type=single-node",
           "-Eplugins.security.disabled=true", "-Ehttp.port=9200"]
    log = Path("/content/opensearch.log")
    subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(90):
        if os_alive():
            print("OpenSearch is up:", client.info()["version"]["number"])
            return
        time.sleep(2)
    raise RuntimeError(f"OpenSearch did not start:\n{log.read_text()[-3000:]}")


start_opensearch()

max_w = max(BM25F_TUNED.w)
REPS = {f: max(1, round(WCONCAT_MAX_REPS * w / max_w)) for f, w in zip(FIELDS, BM25F_TUNED.w)}
OS_INDEX = "esci_" + hashlib.md5(json.dumps([cache_id, REPS, round(BM25F_TUNED.k1, 3), EMBED_MODEL, DIM, HNSW_M,
                                             HNSW_EF_CONSTRUCTION, KEYWORD_MAX_CHARS]).encode()).hexdigest()[:10]
print("Weighted concatenation repeats:", REPS, "| index:", OS_INDEX)


def clean_text(v):
    return "" if v is None or (isinstance(v, float) and math.isnan(v)) else str(v)


def os_index_body(with_ef_search=True):
    hnsw = {"m": HNSW_M, "ef_construction": HNSW_EF_CONSTRUCTION}
    if with_ef_search:
        hnsw["ef_search"] = K_DENSE
    text = {"type": "text", "analyzer": "english"}
    return {
        "settings": {"index": {
            "number_of_shards": 1, "number_of_replicas": 0, "refresh_interval": "-1", "knn": True,
            "max_result_window": max(K_LEX, K_DENSE, EVAL_DEPTH) + 1000,
            "similarity": {"wconcat": {"type": "BM25", "k1": round(BM25F_TUNED.k1, 4), "b": BM25_B}}}},
        "mappings": {"_source": {"enabled": False}, "properties": {
            **{f: dict(text) for f in FIELDS},
            "weighted_all": {**text, "similarity": "wconcat"},
            **{FACET[f]["os_field"]: {"type": "keyword", "ignore_above": KEYWORD_MAX_CHARS} for f in FACET_FIELDS},
            "vec": {"type": "knn_vector", "dimension": DIM, "method": {
                "name": "hnsw", "engine": "faiss", "space_type": "innerproduct", "parameters": hnsw}},
        }},
    }


def os_actions():
    cols = [COLUMN_OF[f] for f in FIELDS]
    for start in range(0, index.N, 10_000):
        chunk = products.iloc[start:start + 10_000][cols]
        vecs = np.round(EMB_NP[start:start + len(chunk)].astype(np.float32), 5)
        for i, row in enumerate(chunk.itertuples(index=False, name=None)):
            doc = start + i
            src = dict(zip(FIELDS, map(clean_text, row)))
            src["weighted_all"] = " ".join(" ".join([src[f]] * REPS[f]) for f in FIELDS if src[f])
            for f in FACET_FIELDS:
                if FACET[f]["values"][doc]:
                    src[FACET[f]["os_field"]] = FACET[f]["values"][doc]
            src["vec"] = vecs[i].tolist()
            yield {"_index": OS_INDEX, "_id": str(doc), "_source": src}


def build_os_index():
    if client.indices.exists(index=OS_INDEX):
        client.indices.refresh(index=OS_INDEX)
        if client.count(index=OS_INDEX)["count"] == index.N:
            print(f"Reusing {OS_INDEX}")
            return
        client.indices.delete(index=OS_INDEX)
    client.cluster.put_settings(body={"persistent": {"knn.algo_param.index_thread_qty": max(1, (os.cpu_count() or 2) // 2)}})
    try:
        client.indices.create(index=OS_INDEX, body=os_index_body())
    except Exception as err:
        if "ef_search" not in str(err):
            raise
        print("Mapping rejected ef_search; queries pass it as method_parameters instead")
        client.indices.create(index=OS_INDEX, body=os_index_body(with_ef_search=False))
    t0, failed = time.perf_counter(), 0
    for i, (ok, _) in enumerate(helpers.parallel_bulk(client, os_actions(), thread_count=4, chunk_size=500,
                                                      max_chunk_bytes=50 * 2**20, request_timeout=600)):
        failed += not ok
        if i and i % 200_000 == 0:
            print(f"  indexed {i:,} ({time.perf_counter() - t0:.0f}s)")
    client.indices.put_settings(index=OS_INDEX, body={"index": {"refresh_interval": None}})
    client.indices.refresh(index=OS_INDEX, request_timeout=3600)
    print(f"Bulk indexing: {time.perf_counter() - t0:.0f}s, {failed} failures")
    if OS_FORCE_MERGE:
        t1 = time.perf_counter()
        client.indices.forcemerge(index=OS_INDEX, max_num_segments=1, request_timeout=7200)
        print(f"Force merge (rebuilds the HNSW graph): {time.perf_counter() - t1:.0f}s")
    n = client.count(index=OS_INDEX)["count"]
    assert n == index.N and failed == 0, f"indexed {n:,} of {index.N:,}"


build_os_index()
try:
    client.transport.perform_request("GET", f"/_plugins/_knn/warmup/{OS_INDEX}")
except Exception as err:
    print(f"k-NN warmup skipped ({type(err).__name__})")

OS_KNN_METHOD_PARAMS = True
OS_FILTER_PATH = ["took", "responses.took", "responses.error", "responses.hits.total.value",
                  "responses.hits.hits._id", "responses.hits.hits._score",
                  "responses.aggregations.*.buckets.key", "responses.aggregations.*.buckets.doc_count"]


def os_knn(vec, k, flt=None):
    clause = {"vector": vec, "k": k}
    if OS_KNN_METHOD_PARAMS:
        clause["method_parameters"] = {"ef_search": max(k, 100)}
    if flt is not None:
        clause["filter"] = flt
    return {"knn": {"vec": clause}}


def os_ip_from_score(s):
    # faiss innerproduct _score: ip + 1 when ip >= 0, else 1 / (1 - ip).
    s = np.asarray(s, dtype=np.float64)
    return np.where(s >= 1.0, s - 1.0, 1.0 - 1.0 / np.maximum(s, 1e-12))


def os_msearch(bodies):
    payload = []
    for b in bodies:
        payload += [{"index": OS_INDEX}, b]
    resp = client.msearch(body=payload, filter_path=OS_FILTER_PATH, request_timeout=600)
    for r in resp["responses"]:
        if "error" in r:
            raise RuntimeError(str(r["error"])[:500])
    return resp


def parse_hits(r):
    hits = r.get("hits", {}).get("hits", [])
    ids = np.fromiter((int(h["_id"]) for h in hits), dtype=np.int64, count=len(hits))
    return ids, np.fromiter((h["_score"] for h in hits), dtype=np.float64, count=len(hits))


# ---- Check: k-NN options and the score conversion, against exact similarities -----------------------
pv = probe[:1].float().cpu().numpy()[0]
try:
    r = client.search(index=OS_INDEX, body={"query": os_knn(pv.tolist(), 100), "size": 100, "_source": False})
except Exception as err:
    if "method_parameters" not in str(err):
        raise
    OS_KNN_METHOD_PARAMS = False
    print("This version rejects method_parameters; using the mapping's ef_search")
    r = client.search(index=OS_INDEX, body={"query": os_knn(pv.tolist(), 100), "size": 100, "_source": False})
hit_ids, hit_sc = parse_hits(r)
err = np.abs(os_ip_from_score(hit_sc) - EMB_NP[hit_ids].astype(np.float32) @ pv).max()
ref_ids, _ = DENSE.exact(probe[:1], 100)
assert err < 0.01, f"k-NN score conversion is off by {err:.4f}"
print(f"CHECK PASSED: OpenSearch k-NN scores convert to inner products (max error {err:.4f}); "
      f"HNSW overlap with exact top 100: {len(set(hit_ids) & set(ref_ids[0].cpu().numpy())) / 100:.2f}")

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/opensearchpy/connection/http_urllib3.py", line 280, in perform_request
    response = self.pool.urlopen(
        method, url, body, retries=Retry(False), headers=request_headers, **kw
    )
  File "/usr/local/lib/python3.13/dist-packages/urlli

OpenSearch is up: 2.19.1
Weighted concatenation repeats: {'title': 8, 'brand': 1, 'color': 1, 'bullet_point': 1, 'description': 1} | index: esci_3e89c3ae43
  indexed 200,000 (89s)
  indexed 400,000 (186s)
  indexed 600,000 (280s)
  indexed 800,000 (379s)
  indexed 1,000,000 (483s)
  indexed 1,200,000 (585s)
Bulk indexing: 595s, 0 failures
Force merge (rebuilds the HNSW graph): 400s
CHECK PASSED: OpenSearch k-NN scores convert to inner products (max error 0.0000); HNSW overlap with exact top 100: 0.83


## 6 · Experiment 1b · Lexical quality, both systems tuned

OpenSearch gets a random search over `cross_fields` boosts and `tie_breaker` (`N_OS_TRIALS` trials, hand boosts included), selected on the **same selection queries and metric** as the GPU checkpoints. Weighted concatenation reuses the GPU-tuned weights and `k1`.

All rows are then evaluated once on the test queries. Δ is paired against the best stock OpenSearch setting, `cross_fields` tuned.

The **title-only rows are a sanity check**: both engines then run the same BM25 with the same analyzer and per-field statistics, so they should agree within about 0.01.

In [16]:
def os_search(make_query, texts, k, batch=32):
    ids = np.full((len(texts), k), -1, np.int64)
    for s in range(0, len(texts), batch):
        resp = os_msearch([{"query": make_query(t), "size": k, "_source": False, "track_total_hits": False}
                           for t in texts[s:s + batch]])
        for i, r in enumerate(resp["responses"]):
            h, _ = parse_hits(r)
            ids[s + i, :len(h)] = h
    return ids


def os_rerank_scores(make_query, qs, batch=32):
    # Score only each query's judged products; products the query does not match get 0.
    scores = np.zeros(len(qs.q), np.float32)
    order = np.argsort(qs.q, kind="stable")
    bounds = np.searchsorted(qs.q[order], np.arange(len(qs) + 1))
    for s in range(0, len(qs), batch):
        e = min(s + batch, len(qs))
        pair_lists = [order[bounds[i]:bounds[i + 1]] for i in range(s, e)]
        bodies = [{"query": {"bool": {"must": make_query(qs.texts[i]),
                                      "filter": {"ids": {"values": [str(d) for d in qs.doc[pairs]]}}}},
                   "size": len(pairs), "_source": False, "track_total_hits": False}
                  for i, pairs in zip(range(s, e), pair_lists)]
        for pairs, r in zip(pair_lists, os_msearch(bodies)["responses"]):
            ids, sc = parse_hits(r)
            got = dict(zip(ids.tolist(), sc.tolist()))
            scores[pairs] = [got.get(int(d), 0.0) for d in qs.doc[pairs]]
    return scores


def evaluate_os(make_query, qs):
    return {**retrieve_metrics(qs, os_search(make_query, qs.texts, EVAL_DEPTH)),
            **rerank_metrics(qs, os_rerank_scores(make_query, qs))}


def select_score_os(make_query):
    return retrieve_metrics(SEL, os_search(make_query, SEL.texts, 10))["ndcg_10"].mean()


def q_match(field_name):
    return lambda t: {"match": {field_name: t}}


def q_cross(boosts, tie=0.0):
    fields = [f"{f}^{boosts[f]:.4g}" for f in FIELDS]
    return lambda t: {"multi_match": {"query": t, "type": "cross_fields", "fields": fields, "tie_breaker": tie}}


hand_boosts = {f: HAND_WEIGHTS[f] / HAND_WEIGHTS["title"] for f in FIELDS}
trial_rng = np.random.default_rng(SEED + 1)
trials = [(hand_boosts, 0.0)] + [
    ({f: 1.0 if f == "title" else float(np.exp(trial_rng.uniform(np.log(0.03), np.log(3.0)))) for f in FIELDS},
     float(trial_rng.uniform(0.0, 0.5)))
    for _ in range(N_OS_TRIALS - 1)]
t0 = time.perf_counter()
trial_rows = []
for boosts, tie in trials:
    trial_rows.append(dict(**{f"boost_{f}": boosts[f] for f in FIELDS}, tie_breaker=tie,
                           select_ndcg_10=select_score_os(q_cross(boosts, tie))))
trial_df = pd.DataFrame(trial_rows)
best = trial_df["select_ndcg_10"].idxmax()
OS_TUNED = ({f: trial_df.loc[best, f"boost_{f}"] for f in FIELDS}, float(trial_df.loc[best, "tie_breaker"]))
print(f"OpenSearch search: {len(trials)} trials in {time.perf_counter() - t0:.0f}s. Selection nDCG@10: "
      f"hand {trial_df.loc[0, 'select_ndcg_10']:.4f} -> tuned {trial_df.loc[best, 'select_ndcg_10']:.4f}")
display(trial_df.sort_values("select_ndcg_10", ascending=False).head(5).round(3))

OpenSearch search: 30 trials in 53s. Selection nDCG@10: hand 0.2778 -> tuned 0.2778


,boost_title,boost_brand,boost_color,boost_bullet_point,boost_description,tie_breaker,select_ndcg_10
0,1.0000,0.6670,0.3330,0.3330,0.1670,0.0000,0.2780
22,1.0000,0.5550,0.0520,0.3380,0.1870,0.0970,0.2770
20,1.0000,0.0640,0.5540,0.3010,0.0370,0.1210,0.2730
11,1.0000,1.4690,0.2170,0.0370,0.0860,0.0370,0.2720
29,1.0000,0.5380,0.0340,0.0370,0.0600,0.2210,0.2710


In [17]:
LEXICAL = {
    "GPU title only": ("gpu", Params.of({"title": 1.0})),
    "GPU BM25F, hand weights": ("gpu", HAND),
    "GPU BM25F, tuned": ("gpu", BM25F_TUNED),
    "OS title only": ("os", q_match("title")),
    "OS cross_fields, hand boosts": ("os", q_cross(hand_boosts, 0.0)),
    "OS cross_fields, tuned": ("os", q_cross(*OS_TUNED)),
    "OS weighted concatenation": ("os", q_match("weighted_all")),
}
lex_results, lex_select = {}, {}
for name, (engine, cfg) in LEXICAL.items():
    t0 = time.perf_counter()
    lex_select[name] = select_score_gpu(cfg) if engine == "gpu" else select_score_os(cfg)
    lex_results[name] = evaluate_gpu(TEST, cfg) if engine == "gpu" else evaluate_os(cfg, TEST)
    print(f"{name}: {time.perf_counter() - t0:.0f}s")

BASE_LEX = "OS cross_fields, tuned"
print(f"\nWhole-catalog retrieval, {len(TEST):,} test queries (unjudged = 0 gain). Δ paired against {BASE_LEX!r}.")
for m in ("ndcg_10", "recall_100", "recall_1000"):
    display(compare(lex_results, m, BASE_LEX))
print("Rerank judged products (ESCI Task 1). rerank_ndcg_full is comparable with published numbers.")
for m in ("rerank_ndcg_10", "rerank_ndcg_full"):
    display(compare(lex_results, m, BASE_LEX))

# ---- Check: single-field BM25 agrees across engines -------------------------------------------------
for m in ("ndcg_10", "rerank_ndcg_10"):
    gap = lex_results["OS title only"][m].mean() - lex_results["GPU title only"][m].mean()
    status = "CHECK PASSED" if abs(gap) < 0.01 else "WARNING (analyzer or statistics differ)"
    print(f"{status}: title-only {m}, OpenSearch minus GPU = {gap:+.4f}")

# The pipeline uses each engine's best lexical setting, chosen on the selection queries only.
os_rows = ["OS cross_fields, hand boosts", "OS cross_fields, tuned", "OS weighted concatenation"]
OS_PIPE_NAME = max(os_rows, key=lex_select.get)
OS_PIPE_QUERY = LEXICAL[OS_PIPE_NAME][1]
print(f"\nPipeline lexical settings (picked on selection queries): GPU 'GPU BM25F, tuned' | OpenSearch {OS_PIPE_NAME!r}")
display(pd.Series(lex_select, name="selection nDCG@10").round(4).to_frame())
RESULTS["lexical"] = {n: {m: float(v.mean()) for m, v in r.items()} for n, r in lex_results.items()}
RESULTS["lexical_selection"] = lex_select

GPU title only: 3s
GPU BM25F, hand weights: 3s
GPU BM25F, tuned: 3s
OS title only: 36s
OS cross_fields, hand boosts: 69s
OS cross_fields, tuned: 69s
OS weighted concatenation: 38s

Whole-catalog retrieval, 8,956 test queries (unjudged = 0 gain). Δ paired against 'OS cross_fields, tuned'.


,ndcg_10,"Δ vs OS cross_fields, tuned",95% low,95% high
GPU title only,0.2617,-0.0082,-0.0100,-0.0063
"GPU BM25F, hand weights",0.2961,0.0262,0.0233,0.0292
"GPU BM25F, tuned",0.3039,0.0341,0.0312,0.0369
OS title only,0.2619,-0.0080,-0.0097,-0.0062
"OS cross_fields, hand boosts",0.2698,0.0000,0.0000,0.0000
"OS cross_fields, tuned",0.2698,0.0000,0.0000,0.0000
OS weighted concatenation,0.2906,0.0208,0.0174,0.0241


,recall_100,"Δ vs OS cross_fields, tuned",95% low,95% high
GPU title only,0.4803,-0.0264,-0.0293,-0.0233
"GPU BM25F, hand weights",0.5356,0.0289,0.0257,0.0321
"GPU BM25F, tuned",0.5338,0.0271,0.0237,0.0305
OS title only,0.4794,-0.0273,-0.0301,-0.0242
"OS cross_fields, hand boosts",0.5067,0.0000,0.0000,0.0000
"OS cross_fields, tuned",0.5067,0.0000,0.0000,0.0000
OS weighted concatenation,0.5015,-0.0052,-0.0096,-0.0010


,recall_1000,"Δ vs OS cross_fields, tuned",95% low,95% high
GPU title only,0.7017,-0.0329,-0.0356,-0.0303
"GPU BM25F, hand weights",0.7464,0.0118,0.0092,0.0143
"GPU BM25F, tuned",0.7415,0.0069,0.0042,0.0096
OS title only,0.7016,-0.0330,-0.0356,-0.0303
"OS cross_fields, hand boosts",0.7346,0.0000,0.0000,0.0000
"OS cross_fields, tuned",0.7346,0.0000,0.0000,0.0000
OS weighted concatenation,0.7089,-0.0257,-0.0292,-0.0222


Rerank judged products (ESCI Task 1). rerank_ndcg_full is comparable with published numbers.


,rerank_ndcg_10,"Δ vs OS cross_fields, tuned",95% low,95% high
GPU title only,0.7013,-0.0097,-0.0113,-0.0082
"GPU BM25F, hand weights",0.7155,0.0045,0.0024,0.0065
"GPU BM25F, tuned",0.7174,0.0063,0.0044,0.0083
OS title only,0.7014,-0.0096,-0.0112,-0.0082
"OS cross_fields, hand boosts",0.7111,0.0000,0.0000,0.0000
"OS cross_fields, tuned",0.7111,0.0000,0.0000,0.0000
OS weighted concatenation,0.7155,0.0044,0.0020,0.0067


,rerank_ndcg_full,"Δ vs OS cross_fields, tuned",95% low,95% high
GPU title only,0.8244,-0.0059,-0.0070,-0.0049
"GPU BM25F, hand weights",0.8333,0.0030,0.0015,0.0045
"GPU BM25F, tuned",0.8351,0.0048,0.0034,0.0062
OS title only,0.8245,-0.0058,-0.0069,-0.0048
"OS cross_fields, hand boosts",0.8303,0.0000,0.0000,0.0000
"OS cross_fields, tuned",0.8303,0.0000,0.0000,0.0000
OS weighted concatenation,0.8330,0.0026,0.0010,0.0042


CHECK PASSED: title-only ndcg_10, OpenSearch minus GPU = +0.0002
CHECK PASSED: title-only rerank_ndcg_10, OpenSearch minus GPU = +0.0001

Pipeline lexical settings (picked on selection queries): GPU 'GPU BM25F, tuned' | OpenSearch 'OS weighted concatenation'


,selection nDCG@10
GPU title only,0.2681
"GPU BM25F, hand weights",0.3022
"GPU BM25F, tuned",0.3119
OS title only,0.2696
"OS cross_fields, hand boosts",0.2778
"OS cross_fields, tuned",0.2778
OS weighted concatenation,0.2955


## 7 · Experiment 2 · Full pipeline: retrieve 20k, rank with a GBDT, facets, pagination

Both systems serve the same request:

1. Encode the query (same model, same GPU) and extract query features.
2. Retrieve from the whole catalog: lexical top `K_LEX` and dense top `K_DENSE`.
3. Fuse with RRF and keep `K_CAND` candidates.
4. Score every candidate with XGBoost on the GPU and keep the top `K_FINAL`.
5. Count brand and color facets over the full lexical match set.
6. Return page 1. Pages 2 onward come from a cache of the top `K_FINAL`.

| | GPU engine | OpenSearch + GPU ranker |
|---|---|---|
| Lexical | BM25F, tuned | Best OpenSearch setting from experiment 1 |
| Dense | cuVS | faiss HNSW, `ef_search = k` |
| Facets | GPU `bincount` over the match mask | `terms` aggregations |
| Fusion, features | On the GPU | On the CPU after parsing JSON, then copied to the GPU |

Three rows, each with its own GBDT trained the same way:

- **OpenSearch + GPU ranker**
- **GPU, same features:** identical feature definitions to the OpenSearch path, so any gap comes from retrieval and architecture.
- **GPU, complete features:** adds every candidate's lexical and dense score, which a co-located engine gets for free.

In [18]:
NAN = float("nan")


class Timer:
    def __init__(self):
        self.laps, self.extra, self.t = {}, {}, time.perf_counter()

    def lap(self, name):
        if IS_CUDA:
            torch.cuda.synchronize()
        now = time.perf_counter()
        self.laps[name] = self.laps.get(name, 0.0) + (now - self.t) * 1000
        self.t = now


class Ranker:
    def __init__(self, booster):
        self.booster = booster
        best = getattr(booster, "best_iteration", None)
        self.iters = (0, best + 1) if best is not None else (0, 0)

    def predict(self, X):
        # Torch input stays on the GPU; NumPy input (the OpenSearch path) is copied to the GPU here.
        is_t = torch.is_tensor(X)
        if len(X) == 0:
            return torch.empty(0, device=DEVICE) if is_t else np.empty(0, np.float32)
        if IS_CUDA:
            data = cp.from_dlpack(X.contiguous()) if is_t else cp.asarray(X)
            out = self.booster.inplace_predict(data, iteration_range=self.iters)
            return torch.as_tensor(out, device=DEVICE) if is_t else cp.asnumpy(out)
        out = self.booster.inplace_predict(X.cpu().numpy() if is_t else X, iteration_range=self.iters)
        return torch.as_tensor(out, device=DEVICE) if is_t else out


# ------------------------------------------------------------------------------- GPU engine
GPU_BUF = {n: torch.empty(MAX_BATCH, index.N, dtype=torch.float32, device=DEVICE)
           for n in ("lex", "rrf", "lrank", "drank", "dval")}


def gpu_filter(filters, B):
    if not filters or all(f is None for f in filters):
        return None
    m = torch.ones(B, index.N, dtype=torch.bool, device=DEVICE)
    for b, flt in enumerate(filters):
        if flt is not None:
            m[b] = FACET[flt[0]]["codes_t"] == FACET[flt[0]]["code_of"].get(flt[1], -1)
    return m


def gpu_facets(match):
    out = {}
    for f in FACET_FIELDS:
        fc, ids, vals = FACET[f], [], []
        for b in range(match.shape[0]):
            cnt = torch.bincount(fc["codes_t"][match[b] & fc["present_t"]], minlength=fc["n"])
            v, i = cnt.topk(min(FACET_SIZE, fc["n"]))
            ids.append(i)
            vals.append(v)
        out[f] = (torch.stack(ids), torch.stack(vals))
    return out


def facets_to_host(fac, B):
    out = [{} for _ in range(B)]
    for f, (ids, vals) in fac.items():
        ids, vals = ids.cpu().numpy(), vals.cpu().numpy()
        for b in range(B):
            out[b][f] = {FACET[f]["names"][i]: int(v) for i, v in zip(ids[b], vals[b]) if v > 0}
    return out


def match_matrix(code_lists, f):
    m = torch.zeros(len(code_lists), FACET[f]["n"], dtype=torch.bool, device=DEVICE)
    rows = np.concatenate([np.full(len(c), b, dtype=np.int64) for b, c in enumerate(code_lists)])
    cols = np.concatenate(code_lists).astype(np.int64)
    if len(cols):
        m[torch.as_tensor(rows, device=DEVICE), torch.as_tensor(cols, device=DEVICE)] = True
    return m


class GpuEngine:
    def __init__(self, name, complete):
        self.name, self.complete, self.ranker, self.need_terms = name, complete, None, True

    @torch.no_grad()
    def stage1(self, qv, info, filters=None, timer=None):
        timer = timer or Timer()
        B, N = len(info), index.N
        filt = gpu_filter(filters, B)
        lex = GPU_BUF["lex"][:B]
        index.score_into(lex, [x["terms"] for x in info], BM25F_TUNED)
        if filt is not None:
            lex.masked_fill_(~filt, 0.0)
        match = lex > 0
        n_match = match.sum(1)
        facets = gpu_facets(match)
        lv, li = lex.topk(min(K_LEX, N), dim=1)
        li.masked_fill_(lv <= 0, -1)
        timer.lap("lexical + facets")

        di, dsim = DENSE.search(qv, K_DENSE, filt)
        timer.lap("dense")

        rows = torch.arange(B, device=DEVICE)[:, None]
        rrf, lrank, drank, dval = (GPU_BUF[n][:B] for n in ("rrf", "lrank", "drank", "dval"))
        rrf.zero_(), lrank.zero_(), drank.zero_()
        for ids, rank_buf in ((li, lrank), (di, drank)):
            ok = ids >= 0
            r = torch.arange(1, ids.shape[1] + 1, device=DEVICE, dtype=torch.float32).expand_as(ids)
            flat = (rows * N + ids)[ok]
            rank_buf.view(-1)[flat] = r[ok]
            rrf.view(-1).index_add_(0, flat, 1.0 / (RRF_K + r[ok]))
        ok = di >= 0
        dval.view(-1)[(rows * N + di)[ok]] = dsim[ok]
        cv, ci = rrf.topk(min(K_CAND, N), dim=1)
        ci.masked_fill_(cv <= 0, -1)
        timer.lap("fusion")

        safe, valid = ci.clamp_min(0), ci >= 0
        lr, dr = lrank.gather(1, safe), drank.gather(1, safe)
        in_l, in_d = lr > 0, dr > 0
        if self.complete:     # Every candidate's scores, read from the full buffers
            lex_f = lex.gather(1, safe)
            dense_f = (qv @ EMB.T).float().gather(1, safe)
        else:                 # Only what the retrievers returned, exactly what OpenSearch hands back
            lex_f = torch.where(in_l, lex.gather(1, safe), NAN)
            dense_f = torch.where(in_d, dval.gather(1, safe), NAN)
        lex_top = torch.where(lv[:, :1] > 0, lv[:, :1], NAN)
        dense_top = torch.where(di[:, :1] >= 0, dsim[:, :1], NAN)
        bm = match_matrix([x["brand"] for x in info], "brand")[rows, FACET["brand"]["codes_t"][safe]]
        cm = match_matrix([x["color"] for x in info], "color")[rows, FACET["color"]["codes_t"][safe]]
        q_len = torch.tensor([float(x["n_tok"]) for x in info], device=DEVICE)[:, None].expand_as(cv)
        n_log = torch.log1p(n_match.float())[:, None].expand_as(cv)
        cols = [lex_f, lex_f / lex_top, torch.where(in_l, torch.log1p(lr), NAN), dense_f, dense_f - dense_top,
                torch.where(in_d, torch.log1p(dr), NAN), cv, (in_l & in_d).float(), bm.float(), cm.float()]
        X = torch.cat([torch.stack(cols, dim=2), STATIC_T[safe], torch.stack([q_len, n_log], dim=2)], dim=2)
        timer.lap("features")
        return dict(ids=ci, valid=valid, rrf=cv, X=X, facets=facets, n_match=n_match, li=li, lv=lv, di=di, dsim=dsim)

    @torch.no_grad()
    def rank(self, s1, timer=None):
        timer = timer or Timer()
        ids, valid = s1["ids"], s1["valid"]
        if self.ranker is None:
            score = torch.where(valid, s1["rrf"], float("-inf"))
        else:
            score = torch.full(ids.shape, float("-inf"), device=DEVICE)
            score[valid] = self.ranker.predict(s1["X"][valid]).float()
        v, o = score.topk(min(K_FINAL, ids.shape[1]), dim=1)
        top = ids.gather(1, o).masked_fill(torch.isinf(v), -1)
        timer.lap("ranker")
        out = dict(ranked=list(top.cpu().numpy()), facets=facets_to_host(s1["facets"], ids.shape[0]),
                   n_match=s1["n_match"].cpu().numpy())
        timer.lap("results to host")
        return out

    def candidates(self, s1):
        return list(s1["ids"].cpu().numpy())

    def rows(self, s1, b, sel):
        return s1["X"][b, torch.as_tensor(sel, device=DEVICE)].cpu().numpy()


# ------------------------------------------------------------------------- OpenSearch engine
def os_filter(flt):
    return None if flt is None else {"term": {FACET[flt[0]]["os_field"]: flt[1]}}


def rank_of(cand, ids):
    # 1-based position of each candidate in `ids`, 0 when absent.
    if len(ids) == 0:
        return np.zeros(len(cand), dtype=np.int64)
    order = np.argsort(ids, kind="stable")
    s = ids[order]
    pos = np.searchsorted(s, cand).clip(max=len(s) - 1)
    return np.where(s[pos] == cand, order[pos] + 1, 0)


def host_fuse(lex_ids, dense_ids):
    w = np.concatenate([1.0 / (RRF_K + np.arange(1, len(lex_ids) + 1)), 1.0 / (RRF_K + np.arange(1, len(dense_ids) + 1))])
    uniq, inv = np.unique(np.concatenate([lex_ids, dense_ids]).astype(np.int64), return_inverse=True)
    rrf = np.bincount(inv.reshape(-1), weights=w, minlength=len(uniq))
    order = np.argsort(-rrf, kind="stable")[:K_CAND]
    return uniq[order], rrf[order]


def host_features(x, lex_ids, lex_sc, dense_ids, dense_sim, n_match):
    # Same definitions as GpuEngine.stage1 with complete=False, written independently in NumPy.
    cand, rrf = host_fuse(lex_ids, dense_ids)
    lr, dr = rank_of(cand, lex_ids), rank_of(cand, dense_ids)
    in_l, in_d = lr > 0, dr > 0
    lex_f = np.full(len(cand), np.nan)
    lex_f[in_l] = lex_sc[lr[in_l] - 1]
    dense_f = np.full(len(cand), np.nan)
    dense_f[in_d] = dense_sim[dr[in_d] - 1]
    lex_top = lex_sc[0] if len(lex_sc) and lex_sc[0] > 0 else np.nan
    dense_top = dense_sim[0] if len(dense_sim) else np.nan
    with np.errstate(invalid="ignore", divide="ignore"):
        cols = [lex_f, lex_f / lex_top, np.where(in_l, np.log1p(lr), np.nan), dense_f, dense_f - dense_top,
                np.where(in_d, np.log1p(dr), np.nan), rrf, (in_l & in_d).astype(np.float64),
                np.isin(FACET["brand"]["codes"][cand], x["brand"]), np.isin(FACET["color"]["codes"][cand], x["color"])]
    X = np.column_stack(cols + [STATIC[cand], np.full(len(cand), float(x["n_tok"])),
                                np.full(len(cand), np.log1p(n_match))]).astype(np.float32)
    return cand, rrf, X


class OsEngine:
    def __init__(self, name="OpenSearch + GPU ranker"):
        self.name, self.ranker, self.need_terms = name, None, False

    def stage1(self, qv, info, filters=None, timer=None):
        timer = timer or Timer()
        vecs = qv.float().cpu().numpy()
        aggs = {f: {"terms": {"field": FACET[f]["os_field"], "size": FACET_SIZE}} for f in FACET_FIELDS}
        bodies = []
        for i, x in enumerate(info):
            flt = os_filter(filters[i]) if filters else None
            lexq = OS_PIPE_QUERY(x["text"])
            if flt is not None:
                lexq = {"bool": {"must": [lexq], "filter": [flt]}}
            bodies += [{"query": lexq, "size": K_LEX, "_source": False, "track_total_hits": True, "aggs": aggs},
                       {"query": os_knn(vecs[i].tolist(), K_DENSE, flt), "size": K_DENSE, "_source": False}]
        resp = os_msearch(bodies)
        timer.extra["opensearch took"] = timer.extra.get("opensearch took", 0.0) + float(resp.get("took", np.nan))
        timer.lap("opensearch (incl. JSON)")

        cands, rrfs, Xs, facets, n_match = [], [], [], [], []
        rs = resp["responses"]
        for i, x in enumerate(info):
            lex_r, knn_r = rs[2 * i], rs[2 * i + 1]
            lex_ids, lex_sc = parse_hits(lex_r)
            d_ids, d_sc = parse_hits(knn_r)
            total = lex_r.get("hits", {}).get("total", {}).get("value", 0)
            cand, rrf, X = host_features(x, lex_ids, lex_sc, d_ids, os_ip_from_score(d_sc), total)
            cands.append(cand), rrfs.append(rrf), Xs.append(X), n_match.append(total)
            ag = lex_r.get("aggregations", {})
            facets.append({f: {b["key"]: int(b["doc_count"]) for b in ag.get(f, {}).get("buckets", [])} for f in FACET_FIELDS})
        timer.lap("fusion + features (CPU)")
        return dict(cands=cands, rrf=rrfs, X=np.concatenate(Xs) if Xs else np.empty((0, len(FEATURES)), np.float32),
                    offsets=np.cumsum([0] + [len(c) for c in cands]), facets=facets, n_match=np.array(n_match))

    def rank(self, s1, timer=None):
        timer = timer or Timer()
        scores = self.ranker.predict(s1["X"]) if self.ranker is not None else np.concatenate(s1["rrf"] or [[]])
        ranked, off = [], s1["offsets"]
        for b, cand in enumerate(s1["cands"]):
            sc = np.asarray(scores[off[b]:off[b + 1]])
            k = min(K_FINAL, len(cand))
            top = np.argpartition(-sc, k - 1)[:k] if 0 < k < len(cand) else np.arange(k)
            ranked.append(cand[top[np.argsort(-sc[top], kind="stable")]])
        timer.lap("ranker (to GPU and back)")
        return dict(ranked=ranked, facets=s1["facets"], n_match=s1["n_match"])

    def candidates(self, s1):
        return s1["cands"]

    def rows(self, s1, b, sel):
        return s1["X"][s1["offsets"][b] + np.asarray(sel, dtype=np.int64)]


def serve(engine, texts, filters=None, timer=None):
    timer = timer or Timer()
    qv = encode_queries(texts)
    timer.lap("encode query")
    info = query_info(texts, engine.need_terms)
    timer.lap("query features")
    s1 = engine.stage1(qv, info, filters, timer)
    return engine.rank(s1, timer), s1


# ---- Pagination: cache the ranker's top K_FINAL; a miss recomputes the same pipeline ----------------
PAGE_CACHE, PAGE_CACHE_MAX = collections.OrderedDict(), 20_000


def page(engine, text, page_no, flt=None, use_cache=True, timer=None):
    timer = timer or Timer()
    key = (engine.name, text, flt, CATALOG_VERSION)
    hit = PAGE_CACHE.get(key) if use_cache else None
    if hit is None:
        out, _ = serve(engine, [text], [flt] if flt is not None else None, timer)
        ranked = out["ranked"][0]
        hit = (ranked[ranked >= 0], out["facets"][0], int(out["n_match"][0]))
        PAGE_CACHE[key] = hit
        while len(PAGE_CACHE) > PAGE_CACHE_MAX:
            PAGE_CACHE.popitem(last=False)
    PAGE_CACHE.move_to_end(key)
    ids = hit[0][page_no * PAGE_SIZE:(page_no + 1) * PAGE_SIZE]
    out = dict(ids=ids, titles=TITLES[ids].tolist(), facets=hit[1], total=hit[2],
               has_next=(page_no + 1) * PAGE_SIZE < len(hit[0]))
    timer.lap("page")
    return out


OS_ENGINE = OsEngine()
GPU_SAME = GpuEngine("GPU, same features", complete=False)
GPU_FULL = GpuEngine("GPU, complete features", complete=True)
ENGINES = [OS_ENGINE, GPU_SAME, GPU_FULL]
BASELINE = OS_ENGINE.name
print("Engines ready:", [e.name for e in ENGINES])

Engines ready: ['OpenSearch + GPU ranker', 'GPU, same features', 'GPU, complete features']


### Checks before training

- **Fusion and features:** the GPU's lexical and dense lists are fed through the independent NumPy path used for OpenSearch. Candidates, RRF scores and all features must match.
- **Facets:** GPU facet counts must equal a brute-force count over the match set.
- **Cross-engine agreement** (reported, not asserted): match counts and facets differ only through analyzer details.

In [19]:
texts = TEST.texts[:16]
qv = encode_queries(texts)
info = query_info(texts, True)
s1 = GPU_SAME.stage1(qv, info)
worst = collections.defaultdict(float)
for b in range(len(texts)):
    li, lv = s1["li"][b].cpu().numpy(), s1["lv"][b].cpu().numpy().astype(np.float64)
    di, ds = s1["di"][b].cpu().numpy(), s1["dsim"][b].cpu().numpy().astype(np.float64)
    cand_h, rrf_h, X_h = host_features(info[b], li[li >= 0], lv[li >= 0], di[di >= 0], ds[di >= 0], int(s1["n_match"][b]))
    ids_g = s1["ids"][b].cpu().numpy()
    X_g, rrf_g, ids_g = s1["X"][b].cpu().numpy()[ids_g >= 0], s1["rrf"][b].cpu().numpy()[ids_g >= 0], ids_g[ids_g >= 0]
    # Candidate sets may differ only by ties at the K_CAND cut.
    cut = rrf_h.min() if len(rrf_h) else 0.0
    rrf_of = {**dict(zip(ids_g.tolist(), rrf_g.tolist())), **dict(zip(cand_h.tolist(), rrf_h.tolist()))}
    diff = set(cand_h.tolist()) ^ set(ids_g.tolist())
    assert all(abs(rrf_of[d] - cut) < 1e-6 for d in diff), "fused candidate sets differ"
    common, ih, ig = np.intersect1d(cand_h, ids_g, return_indices=True)
    assert np.allclose(rrf_h[ih], rrf_g[ig], rtol=1e-5), "RRF scores differ"
    for j, name in enumerate(FEATURES):
        a, g = X_h[ih, j].astype(np.float64), X_g[ig, j].astype(np.float64)
        ok = np.isclose(a, g, rtol=1e-4, atol=1e-5, equal_nan=True)
        assert ok.all(), f"feature {name} differs: {np.abs(a - g)[~ok][:5]}"
print(f"CHECK PASSED: GPU fusion and all {len(FEATURES)} features match the independent NumPy implementation")

buf = torch.empty(1, index.N, device=DEVICE)
for b in range(4):
    index.score_into(buf, [info[b]["terms"]], BM25F_TUNED)
    m = (buf[0] > 0).cpu().numpy()
    for f in FACET_FIELDS:
        brute = np.bincount(FACET[f]["codes"][m & (FACET[f]["codes"] > 0)], minlength=FACET[f]["n"])
        gpu = facets_to_host({f: s1["facets"][f]}, len(texts))[b][f]
        assert all(brute[FACET[f]["code_of"][k]] == v for k, v in gpu.items()), "facet count differs"
        if gpu:
            brute[[FACET[f]["code_of"][k] for k in gpu]] = 0
            assert brute.max() <= min(gpu.values()), "a larger facet value was left out"
print("CHECK PASSED: GPU facet counts equal a brute-force count over the match set")
del buf, s1

rows = []
for t in texts:
    o, _ = serve(OS_ENGINE, [t])
    g, _ = serve(GPU_SAME, [t])
    row = dict(query=t[:40], os_matches=int(o["n_match"][0]), gpu_matches=int(g["n_match"][0]))
    for f in FACET_FIELDS:
        a, c = o["facets"][0][f], g["facets"][0][f]
        row[f"{f}_facet_jaccard"] = len(set(a) & set(c)) / max(1, len(set(a) | set(c)))
    rows.append(row)
agree = pd.DataFrame(rows)
display(agree)
print("Mean facet agreement across engines:", agree.filter(like="jaccard").mean().round(3).to_dict(),
      "(match sets differ through analyzer details and OpenSearch's lexical setting)")

CHECK PASSED: GPU fusion and all 18 features match the independent NumPy implementation
CHECK PASSED: GPU facet counts equal a brute-force count over the match set


,query,os_matches,gpu_matches,brand_facet_jaccard,color_facet_jaccard
0,1/16th scale gravity wagon without tract,457874,510509,0.5385,1.0000
1,86 inch tv,258138,260218,1.0000,1.0000
2,travel pillow,86285,86532,0.9048,1.0000
3,godiva dark chocolate truffle,53086,53287,1.0000,1.0000
4,syosin resistance bands,168701,169165,0.9048,1.0000
5,progressive press reloading,44190,44345,1.0000,1.0000
6,football headband,15796,15816,1.0000,1.0000
7,ez carafe,2720,2729,1.0000,0.9048
8,kerastase shampoo without sulfates,157890,158055,1.0000,1.0000
9,how not to diet michael greger,75533,76045,0.9048,1.0000


Mean facet agreement across engines: {'brand_facet_jaccard': 0.953, 'color_facet_jaccard': 0.994} (match sets differ through analyzer details and OpenSearch's lexical setting)


### Train one GBDT per engine

Training rows per query: every judged candidate (labels E 3, S 2, C 1, I 0), the top `NEG_TOP` unjudged candidates (hard negatives) and `NEG_RANDOM` random unjudged ones, all labeled 0. Early stopping on validation queries. Each engine trains on its own candidates and features.

In [20]:
_judged = examples.assign(lab=examples["esci_label"].map(LABEL_INT).astype(np.int8)).sort_values(["query_id", "doc"])
JUDGED = {qid: (g["doc"].to_numpy(np.int64), g["lab"].to_numpy()) for qid, g in _judged.groupby("query_id")}


def labels_for(qid, cand):
    docs, labs = JUDGED.get(qid, (np.empty(0, np.int64), np.empty(0, np.int8)))
    if len(docs) == 0:
        return np.zeros(len(cand), np.int8), np.zeros(len(cand), bool)
    pos = np.searchsorted(docs, cand).clip(max=len(docs) - 1)
    found = (docs[pos] == cand) & (cand >= 0)
    return np.where(found, labs[pos], 0).astype(np.int8), found


def select_rows(cand, found, rng):
    unjudged = np.flatnonzero(~found & (cand >= 0))
    rest = unjudged[NEG_TOP:]
    rnd = rng.choice(rest, size=min(NEG_RANDOM, len(rest)), replace=False) if len(rest) else rest
    return np.sort(np.concatenate([np.flatnonzero(found), unjudged[:NEG_TOP], rnd]))


def build_training(engine, qids, rng):
    texts = query_text.loc[qids].tolist()
    Xs, ys, gs = [], [], []
    for s in range(0, len(qids), RUN_BATCH):
        bq, bt = qids[s:s + RUN_BATCH], texts[s:s + RUN_BATCH]
        s1 = engine.stage1(encode_queries(bt), query_info(bt, engine.need_terms))
        for b, (qid, cand) in enumerate(zip(bq, engine.candidates(s1))):
            lab, found = labels_for(qid, cand)
            if found.any():
                sel = select_rows(cand, found, rng)
                Xs.append(engine.rows(s1, b, sel))
                ys.append(lab[sel])
                gs.append(np.full(len(sel), len(gs)))
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(gs)


TRAIN_QIDS = FIT_IDS[:N_TRAIN_QUERIES]
VAL_QIDS = VAL_IDS[N_SELECT:N_SELECT + N_VAL_QUERIES] if len(VAL_IDS) > N_SELECT + 50 else VAL_IDS[:N_VAL_QUERIES]
importance = {}
for eng in ENGINES:
    t0 = time.perf_counter()
    rng = np.random.default_rng(SEED)
    Xtr, ytr, gtr = build_training(eng, TRAIN_QIDS, rng)
    Xva, yva, gva = build_training(eng, VAL_QIDS, rng)
    t1 = time.perf_counter()
    booster = xgb.train({**GBDT_PARAMS, "device": "cuda" if IS_CUDA else "cpu"},
                        xgb.DMatrix(Xtr, label=ytr, qid=gtr), GBDT_ROUNDS,
                        evals=[(xgb.DMatrix(Xva, label=yva, qid=gva), "val")],
                        early_stopping_rounds=GBDT_EARLY_STOP, verbose_eval=False)
    eng.ranker = Ranker(booster)
    gain = booster.get_score(importance_type="gain")
    importance[eng.name] = {FEATURES[int(k[1:])]: v for k, v in gain.items()}
    print(f"{eng.name}: {len(ytr):,} rows from {gtr.max() + 1:,} queries | features {t1 - t0:.0f}s, "
          f"training {time.perf_counter() - t1:.0f}s | {eng.ranker.iters[1]} trees")
print("Feature importance (share of gain):")
display(pd.DataFrame(importance).fillna(0).pipe(lambda d: d / d.sum()).round(3).sort_values(BASELINE, ascending=False))

OpenSearch + GPU ranker: 350,597 rows from 2,979 queries | features 355s, training 1s | 145 trees
GPU, same features: 350,922 rows from 2,979 queries | features 15s, training 1s | 139 trees
GPU, complete features: 350,922 rows from 2,979 queries | features 16s, training 1s | 111 trees
Feature importance (share of gain):


,OpenSearch + GPU ranker,"GPU, same features","GPU, complete features"
in_both,0.8660,0.7950,0.6470
rrf,0.0580,0.1000,0.1720
dense_gap,0.0140,0.0150,0.0400
lex_rank_log,0.0110,0.0160,0.0260
dense_rank_log,0.0070,0.0100,0.0150
lex_norm,0.0060,0.0100,0.0140
lex_score,0.0050,0.0060,0.0110
brand_match,0.0050,0.0060,0.0110
dense_sim,0.0040,0.0050,0.0070
n_match_log,0.0040,0.0050,0.0090


In [21]:
PIPE_TEST = make_query_set("pipeline test", frame_for(TEST_IDS[:N_TEST_QUERIES]), index)


def pad_rows(rows, k):
    out = np.full((len(rows), k), -1, dtype=np.int64)
    for i, r in enumerate(rows):
        r = r[r >= 0][:k]
        out[i, :len(r)] = r
    return out


pipe_results, cand_recall = {}, {}
for eng in ENGINES:
    t0 = time.perf_counter()
    ranked, cands = [], []
    for s in range(0, len(PIPE_TEST), RUN_BATCH):
        out, s1 = serve(eng, PIPE_TEST.texts[s:s + RUN_BATCH])
        ranked += out["ranked"]
        cands += eng.candidates(s1)
    ranked, cands = pad_rows(ranked, K_FINAL), pad_rows(cands, K_CAND)
    pipe_results[eng.name] = retrieve_metrics(PIPE_TEST, ranked)
    pipe_results[f"{eng.name} · RRF order, no GBDT"] = retrieve_metrics(PIPE_TEST, cands[:, :K_FINAL])
    cand_recall[eng.name] = candidate_recall(PIPE_TEST, cands)
    print(f"{eng.name}: {len(PIPE_TEST):,} queries in {time.perf_counter() - t0:.0f}s")

print(f"\nWhole-catalog pipeline, {len(PIPE_TEST):,} test queries (unjudged = 0 gain). Δ paired against {BASELINE!r}.")
for m in ("ndcg_10", "recall_100", "recall_1000"):
    display(compare(pipe_results, m, BASELINE))
print(f"Exact-product recall of the {K_CAND:,} candidates (the ceiling for the ranker):")
display(pd.Series({k: v.mean() for k, v in cand_recall.items()}, name=f"recall@{K_CAND}").round(4).to_frame())
RESULTS["pipeline"] = {n: {m: float(v.mean()) for m, v in r.items()} for n, r in pipe_results.items()}
RESULTS["candidate_recall"] = {k: float(v.mean()) for k, v in cand_recall.items()}

OpenSearch + GPU ranker: 2,000 queries in 208s
GPU, same features: 2,000 queries in 10s
GPU, complete features: 2,000 queries in 10s

Whole-catalog pipeline, 2,000 test queries (unjudged = 0 gain). Δ paired against 'OpenSearch + GPU ranker'.


,ndcg_10,Δ vs OpenSearch + GPU ranker,95% low,95% high
OpenSearch + GPU ranker,0.1667,0.0000,0.0000,0.0000
"OpenSearch + GPU ranker · RRF order, no GBDT",0.3194,0.1527,0.1436,0.1625
"GPU, same features",0.1599,-0.0068,-0.0110,-0.0027
"GPU, same features · RRF order, no GBDT",0.3163,0.1496,0.1403,0.1592
"GPU, complete features",0.1664,-0.0003,-0.0048,0.0040
"GPU, complete features · RRF order, no GBDT",0.3163,0.1496,0.1404,0.1592


,recall_100,Δ vs OpenSearch + GPU ranker,95% low,95% high
OpenSearch + GPU ranker,0.2684,0.0000,0.0000,0.0000
"OpenSearch + GPU ranker · RRF order, no GBDT",0.5890,0.3207,0.3048,0.3360
"GPU, same features",0.2562,-0.0122,-0.0183,-0.0066
"GPU, same features · RRF order, no GBDT",0.5961,0.3278,0.3118,0.3429
"GPU, complete features",0.2572,-0.0112,-0.0171,-0.0055
"GPU, complete features · RRF order, no GBDT",0.5962,0.3278,0.3118,0.3429


,recall_1000,Δ vs OpenSearch + GPU ranker,95% low,95% high
OpenSearch + GPU ranker,0.5712,0.0000,0.0000,0.0000
"OpenSearch + GPU ranker · RRF order, no GBDT",0.8005,0.2294,0.2168,0.2422
"GPU, same features",0.5610,-0.0101,-0.0165,-0.0036
"GPU, same features · RRF order, no GBDT",0.8070,0.2359,0.2238,0.2485
"GPU, complete features",0.5586,-0.0125,-0.0192,-0.0058
"GPU, complete features · RRF order, no GBDT",0.8070,0.2359,0.2238,0.2485


Exact-product recall of the 20,000 candidates (the ceiling for the ranker):


,recall@20000
OpenSearch + GPU ranker,0.9230
"GPU, same features",0.9258
"GPU, complete features",0.9258


## 8 · Experiment 3 · Latency, one request at a time

Only matched pairs, measured end to end from the caller's side.

**3a. Lexical retrieval only:** each engine's selected lexical setting, at depth 10 (a first page) and 1000 (a first stage). The GPU runs in process, so the table also shows **GPU + HTTP**: the GPU time plus a measured localhost HTTP round trip that returns the same number of ids as JSON. That is what a GPU service would add; OpenSearch's numbers already include it.

**3b. Full pipeline:** page 1 with a stage breakdown, pages 2–5 from the cache, a cache miss on page 3, and filtered requests (the user clicks the top color facet).

In [22]:
pct = lambda a: dict(p50=float(np.percentile(a, 50)), p95=float(np.percentile(a, 95)), p99=float(np.percentile(a, 99)))
lat_rng = np.random.default_rng(SEED + 11)
LAT_TEXTS = [TEST.texts[i] for i in lat_rng.choice(len(TEST), size=LAT_QUERIES + LAT_WARMUP, replace=False)]


@torch.no_grad()
def gpu_lexical(texts, k, timer=None):
    timer = timer or Timer()
    buf = GPU_BUF["lex"][:len(texts)]
    index.score_into(buf, [index.term_ids(t) for t in texts], BM25F_TUNED)
    v, i = buf.topk(k, dim=1)
    ids = i.masked_fill(v <= 0, -1).cpu().numpy()
    timer.lap("gpu lexical")
    return ids


def os_lexical(texts, k, timer=None):
    timer = timer or Timer()
    resp = os_msearch([{"query": OS_PIPE_QUERY(t), "size": k, "_source": False, "track_total_hits": False} for t in texts])
    ids = [parse_hits(r)[0] for r in resp["responses"]]
    timer.lap("opensearch lexical")
    return ids


# A small HTTP service in its own process, returning k ids as JSON: the cost of putting the GPU behind HTTP.
ECHO = DATA_DIR / "echo_server.py"
ECHO.write_text('''
import json, sys
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
class H(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    def do_POST(self):
        req = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        body = json.dumps({"ids": [1000000 + i for i in range(req["k"])]}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, *a):
        pass
ThreadingHTTPServer(("127.0.0.1", int(sys.argv[1])), H).serve_forever()
''')
echo_proc = subprocess.Popen([sys.executable, str(ECHO), "8765"])
session = requests.Session()
for _ in range(50):
    try:
        session.post("http://127.0.0.1:8765", json={"q": "x", "k": 1}, timeout=1)
        break
    except Exception:
        time.sleep(0.1)


def http_overhead_ms(k, n):
    out = []
    for t in LAT_TEXTS[:n]:
        t0 = time.perf_counter()
        session.post("http://127.0.0.1:8765", json={"q": t, "k": k}).json()
        out.append((time.perf_counter() - t0) * 1000)
    return np.array(out)


lex_lat_rows = []
for k in (10, EVAL_DEPTH):
    for fn in (gpu_lexical, os_lexical):
        fn(LAT_TEXTS[:LAT_WARMUP], k)
    gpu_ms, os_ms = [], []
    for t in LAT_TEXTS[LAT_WARMUP:]:
        t0 = time.perf_counter(); gpu_lexical([t], k); gpu_ms.append((time.perf_counter() - t0) * 1000)
        t0 = time.perf_counter(); os_lexical([t], k); os_ms.append((time.perf_counter() - t0) * 1000)
    http = http_overhead_ms(k, LAT_QUERIES)
    gpu_ms, os_ms = np.array(gpu_ms), np.array(os_ms)
    lex_lat_rows += [dict(depth=k, engine="GPU BM25F, in process", **pct(gpu_ms)),
                     dict(depth=k, engine="GPU BM25F + HTTP (measured overhead)", **pct(gpu_ms + http[:len(gpu_ms)])),
                     dict(depth=k, engine=f"OpenSearch {OS_PIPE_NAME}", **pct(os_ms))]
echo_proc.terminate()
print("3a · Lexical retrieval latency, batch 1, ms")
display(pd.DataFrame(lex_lat_rows).set_index(["depth", "engine"]).round(2))
RESULTS["latency_lexical"] = lex_lat_rows

3a · Lexical retrieval latency, batch 1, ms


p50     p95     p99
depth engine                                                      
10    GPU BM25F, in process                 1.9500  2.2600  2.6200
      GPU BM25F + HTTP (measured overhead) 42.9600 43.9400 44.2000
      OpenSearch OS weighted concatenation  1.9000  3.0600  3.5700
1000  GPU BM25F, in process                 1.9900  2.4300  2.7700
      GPU BM25F + HTTP (measured overhead) 43.9900 44.6300 45.0300
      OpenSearch OS weighted concatenation 14.1900 19.3900 22.9400

In [23]:
lat_rows, stage_ms = [], {}
for eng in ENGINES:
    for t in LAT_TEXTS[:LAT_WARMUP]:
        page(eng, t, 0, use_cache=False)
    first, later, stages, same = [], [], collections.defaultdict(list), 0
    for t in LAT_TEXTS[LAT_WARMUP:]:
        tm, t0 = Timer(), time.perf_counter()
        p1 = page(eng, t, 0, use_cache=False, timer=tm)
        first.append((time.perf_counter() - t0) * 1000)
        for k_, v in {**tm.laps, **tm.extra}.items():
            stages[k_].append(v)
        seen = list(p1["ids"])
        for p in range(1, N_PAGES):
            t1 = time.perf_counter()
            seen += list(page(eng, t, p)["ids"])
            later.append((time.perf_counter() - t1) * 1000)
        assert len(seen) == len(set(seen)), "pages overlap"                      # No duplicates across pages
        cached = page(eng, t, 2)["ids"]
        same += np.array_equal(cached, page(eng, t, 2, use_cache=False)["ids"])   # Cache miss on page 3
    lat_rows.append(dict(engine=eng.name, **{f"page1_{k_}": v for k_, v in pct(first).items()},
                         **{f"pages2-{N_PAGES}_{k_}": v for k_, v in pct(later).items()},
                         recompute_matches_cache=same / LAT_QUERIES))
    stage_ms[eng.name] = {k_: float(np.median(v)) for k_, v in stages.items()}
print("CHECK PASSED: pages never repeat a product")
print("3b · Pipeline latency, batch 1, ms. recompute_matches_cache below 1.0 means score ties reorder results.")
display(pd.DataFrame(lat_rows).set_index("engine").round(2))
print("Median ms per stage for page 1 ('opensearch took' is OpenSearch's own time, already inside 'opensearch (incl. JSON)'):")
display(pd.DataFrame(stage_ms).round(2))

choices = []
for t in LAT_TEXTS[LAT_WARMUP:]:
    fac = page(GPU_FULL, t, 0)["facets"].get("color", {})
    if fac:
        choices.append((t, ("color", max(fac, key=fac.get))))
flt_rows = []
for eng in ENGINES:
    times, bad = [], 0
    for t, flt in choices:
        t0 = time.perf_counter()
        res = page(eng, t, 0, flt, use_cache=False)
        times.append((time.perf_counter() - t0) * 1000)
        bad += int(np.any(FACET["color"]["values"][res["ids"]] != flt[1]))
    assert bad == 0, f"{eng.name}: {bad} filtered pages contain non-matching products"
    flt_rows.append(dict(engine=eng.name, queries=len(choices), **pct(times)))
print("CHECK PASSED: every filtered page contains only matching products")
print("3c · Filtered request (top color facet clicked), page 1, ms. Dense filtered search on the GPU:",
      "cuVS prefilter" if DENSE.prefilter not in (None, False) else "exact torch")
display(pd.DataFrame(flt_rows).set_index("engine").round(2))
RESULTS["latency_pipeline"] = lat_rows
RESULTS["stage_ms"] = stage_ms
RESULTS["latency_filtered"] = flt_rows

CHECK PASSED: pages never repeat a product
3b · Pipeline latency, batch 1, ms. recompute_matches_cache below 1.0 means score ties reorder results.


,page1_p50,page1_p95,page1_p99,pages2-5_p50,pages2-5_p95,pages2-5_p99,recompute_matches_cache
engine,,,,,,,
OpenSearch + GPU ranker,267.5600,345.1500,378.8800,0.0200,0.0400,0.0500,1.0000
"GPU, same features",20.6400,26.3900,33.8300,0.0200,0.0300,0.0400,0.9800
"GPU, complete features",21.0200,26.7600,34.0900,0.0200,0.0300,0.0400,0.9900


Median ms per stage for page 1 ('opensearch took' is OpenSearch's own time, already inside 'opensearch (incl. JSON)'):


,OpenSearch + GPU ranker,"GPU, same features","GPU, complete features"
encode query,12.6600,11.0900,11.0700
query features,0.6700,0.5800,0.5700
opensearch (incl. JSON),226.3900,NaN,NaN
fusion + features (CPU),21.8100,NaN,NaN
ranker (to GPU and back),4.2200,NaN,NaN
page,0.0700,0.0600,0.0600
opensearch took,190.0000,NaN,NaN
lexical + facets,NaN,2.9700,2.9600
dense,NaN,1.0600,1.0600
fusion,NaN,1.5100,1.5100


CHECK PASSED: every filtered page contains only matching products
3c · Filtered request (top color facet clicked), page 1, ms. Dense filtered search on the GPU: exact torch


,queries,p50,p95,p99
engine,,,,
OpenSearch + GPU ranker,299,180.7600,274.9600,299.9700
"GPU, same features",299,20.7400,25.2700,33.3200
"GPU, complete features",299,21.0800,26.3300,33.0500


## 9 · Experiment 4 · Throughput and cost at a p99 target

Each configuration serves batches of `B` queries together (OpenSearch: one `_msearch`; GPU: one batched pass). For each, the notebook picks the largest throughput whose **p99 batch latency stays under the target**, then converts it to cost per million queries:

- **GPU engine:** `GPU_HOURLY / QPS`. The GPU instance includes its CPUs.
- **OpenSearch:** this machine's vCPUs at `VCPU_HOURLY`, divided by QPS. Client-side work (JSON parsing, fusion, features) ran on the same CPUs during the measurement, so it is included. The pipeline also uses the GPU for the query encoder and the ranker; that GPU time per query is charged at `GPU_HOURLY`, assuming a shared, fully used GPU.

Prices are inputs in the settings cell. OpenSearch shares this machine with the notebook, so its throughput here is a lower bound for a dedicated node.

In [24]:
def sweep(run_batch, texts):
    rows = []
    for B in SWEEP_BATCHES:
        batches = [[texts[(r * B + j) % len(texts)] for j in range(B)] for r in range(SWEEP_ROUNDS)]
        run_batch(batches[0], Timer())
        times, laps = [], collections.Counter()
        for bt in batches:
            tm, t0 = Timer(), time.perf_counter()
            run_batch(bt, tm)
            times.append((time.perf_counter() - t0) * 1000)
            laps.update(tm.laps)
        ms = np.array(times)
        rows.append(dict(batch=B, qps=B * len(ms) / ms.sum() * 1000, p50_ms=np.percentile(ms, 50),
                         p99_ms=np.percentile(ms, 99), stage_ms_per_query={k: v / (B * len(ms)) for k, v in laps.items()}))
    return rows


def at_slo(rows, slo):
    ok = [r for r in rows if r["p99_ms"] <= slo]
    return max(ok, key=lambda r: r["qps"]) if ok else None


def gpu_ms_per_query(r):
    return sum(v for k, v in r["stage_ms_per_query"].items() if k.startswith(("encode", "ranker")))


VCPUS = os.cpu_count() or 1
configs = {
    ("lexical", "GPU BM25F"): (lambda bt, tm: gpu_lexical(bt, EVAL_DEPTH, tm), "gpu"),
    ("lexical", f"OpenSearch {OS_PIPE_NAME}"): (lambda bt, tm: os_lexical(bt, EVAL_DEPTH, tm), "os"),
}
for eng in ENGINES:
    configs[("pipeline", eng.name)] = ((lambda e: lambda bt, tm: serve(e, bt, timer=tm))(eng), "os" if eng is OS_ENGINE else "gpu")

sweep_rows, cost_rows = [], []
for (kind, name), (fn, hw) in configs.items():
    rows = sweep(fn, LAT_TEXTS)
    for r in rows:
        sweep_rows.append(dict(kind=kind, config=name, **{k: v for k, v in r.items() if k != "stage_ms_per_query"}))
    best = at_slo(rows, SLO_P99_MS[kind])
    if best is None:
        cost_rows.append(dict(kind=kind, config=name, slo_ms=SLO_P99_MS[kind], batch=None, qps=0.0, usd_per_million=np.nan))
        continue
    if hw == "gpu":
        usd = GPU_HOURLY / 3600 / best["qps"] * 1e6
    else:
        usd = VCPUS * VCPU_HOURLY / 3600 / best["qps"] * 1e6
        if kind == "pipeline":
            usd += GPU_HOURLY / 3.6e6 * gpu_ms_per_query(best) * 1e6
    cost_rows.append(dict(kind=kind, config=name, slo_ms=SLO_P99_MS[kind], batch=best["batch"], qps=best["qps"],
                          p99_ms=best["p99_ms"], usd_per_million=usd))

print("Throughput sweep")
display(pd.DataFrame(sweep_rows).set_index(["kind", "config", "batch"]).round(1))
print(f"Cost at the p99 target ({VCPUS} vCPUs at ${VCPU_HOURLY}/h for OpenSearch, GPU at ${GPU_HOURLY}/h)")
display(pd.DataFrame(cost_rows).set_index(["kind", "config"]).round(3))
if IS_CUDA:
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
RESULTS["throughput"] = sweep_rows
RESULTS["cost"] = cost_rows

Throughput sweep


qps     p50_ms     p99_ms
kind     config                               batch                                 
lexical  GPU BM25F                            1       530.8000     1.9000     2.0000
                                              4     1,719.0000     2.2000     3.0000
                                              8     2,557.0000     3.2000     4.0000
                                              16    3,597.7000     4.6000     5.4000
                                              32    4,981.5000     6.3000     7.7000
         OpenSearch OS weighted concatenation 1        78.5000    14.1000    18.6000
                                              4       164.7000    23.7000    33.2000
                                              8       215.5000    36.3000    43.5000
                                              16      237.6000    66.8000    74.6000
                                              32      251.6000   127.0000   139.4000
pipeline OpenSearch + GPU ranker              1         4.5000   257.0000   318.6000
                                              4         8.3000   491.6000   586.8000
                                              8         9.3000   857.8000   963.5000
                                              16       10.0000 1,629.7000 1,796.1000
                                              32       10.1000 3,191.5000 3,368.5000
         GPU, same features                   1        49.8000    19.9000    22.0000
                                              4       135.6000    27.3000    36.2000
                                              8       193.9000    42.1000    50.4000
                                              16      274.5000    60.1000    71.4000
                                              32      347.0000    91.3000   108.6000
         GPU, complete features               1        50.1000    19.9000    22.4000
                                              4       139.6000    27.3000    35.1000
                                              8       187.6000    43.8000    51.6000
                                              16      262.6000    61.9000    75.3000
                                              32      336.2000    96.6000   115.0000

Cost at the p99 target (12 vCPUs at $0.0446/h for OpenSearch, GPU at $1.76/h)


slo_ms   batch        qps  p99_ms  usd_per_million
kind     config                                                                                   
lexical  GPU BM25F                             20.0000 32.0000 4,981.5430  7.6700           0.0980
         OpenSearch OS weighted concatenation  20.0000  1.0000    78.5190 18.6260           1.8930
pipeline OpenSearch + GPU ranker              100.0000     NaN     0.0000     NaN              NaN
         GPU, same features                   100.0000 16.0000   274.5270 71.4500           1.7810
         GPU, complete features               100.0000 16.0000   262.5730 75.2560           1.8620

Peak GPU memory: 4.6 GB


## 10 · Summary

In [25]:
lat = pd.DataFrame(lat_rows).set_index("engine")
cost = pd.DataFrame(cost_rows).set_index("config")
summary = pd.DataFrame({
    "lexical nDCG@10": {
        OS_ENGINE.name: lex_results[OS_PIPE_NAME]["ndcg_10"].mean(),
        GPU_SAME.name: lex_results["GPU BM25F, tuned"]["ndcg_10"].mean(),
        GPU_FULL.name: lex_results["GPU BM25F, tuned"]["ndcg_10"].mean()},
    "pipeline nDCG@10": {e.name: pipe_results[e.name]["ndcg_10"].mean() for e in ENGINES},
    "pipeline recall@1000": {e.name: pipe_results[e.name]["recall_1000"].mean() for e in ENGINES},
    "page 1 p99 ms": {e.name: lat.loc[e.name, "page1_p99"] for e in ENGINES},
    "QPS at p99 target": {e.name: cost.loc[e.name, "qps"] for e in ENGINES},
    "$ per 1M queries": {e.name: cost.loc[e.name, "usd_per_million"] for e in ENGINES},
})
display(summary.round(4))
RESULTS["settings"] = dict(DEV=DEV, products=index.N, K_LEX=K_LEX, K_DENSE=K_DENSE, K_CAND=K_CAND, K_FINAL=K_FINAL,
                           embed_model=EMBED_MODEL, dense_backend=DENSE.used, os_version=OS_VERSION,
                           os_pipeline_lexical=OS_PIPE_NAME, bm25f_tuned=BM25F_TUNED.__dict__, os_tuned=OS_TUNED,
                           gpu=torch.cuda.get_device_name() if IS_CUDA else "cpu", vcpus=VCPUS,
                           gpu_hourly=GPU_HOURLY, vcpu_hourly=VCPU_HOURLY)
RESULTS_FILE.write_text(json.dumps(RESULTS, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("Saved", RESULTS_FILE)

,lexical nDCG@10,pipeline nDCG@10,pipeline recall@1000,page 1 p99 ms,QPS at p99 target,$ per 1M queries
OpenSearch + GPU ranker,0.2906,0.1667,0.5712,378.8849,0.0000,NaN
"GPU, same features",0.3039,0.1599,0.5610,33.8343,274.5272,1.7808
"GPU, complete features",0.3039,0.1664,0.5586,34.0874,262.5735,1.8619


Saved /content/esci/results.json


In [26]:

# Diagnose the GBDT: which training recipe helps instead of hurts?
DIAG = make_query_set("diag", frame_for(TEST_IDS[:500]), index)
diag_qids = FIT_IDS[:1500]
eng = GpuEngine("diag", complete=False)
saved = (NEG_TOP, NEG_RANDOM, K_CAND)
recipes = {
    "RRF only (no GBDT)":            None,
    "judged only":                   (0, 0, 20_000),
    "judged + 100 random":           (0, 100, 20_000),
    "current: 50 top + 50 random":   (50, 50, 20_000),
    "current, rerank RRF top 1000":  (50, 50, 1_000),
    "judged only, rerank top 1000":  (0, 0, 1_000),
}
rows = {}
for name, cfg in recipes.items():
    eng.ranker = None
    if cfg is not None:
        NEG_TOP, NEG_RANDOM, K_CAND = cfg
        Xtr, ytr, gtr = build_training(eng, diag_qids, np.random.default_rng(SEED))
        b = xgb.train({**GBDT_PARAMS, "device": "cuda" if IS_CUDA else "cpu"},
                      xgb.DMatrix(Xtr, label=ytr, qid=gtr), 150)
        eng.ranker = Ranker(b)
    ranked = []
    for s in range(0, len(DIAG), RUN_BATCH):
        ranked += serve(eng, DIAG.texts[s:s + RUN_BATCH])[0]["ranked"]
    m = retrieve_metrics(DIAG, pad_rows(ranked, K_FINAL))
    rows[name] = {k: v.mean() for k, v in m.items()}
    NEG_TOP, NEG_RANDOM, K_CAND = saved
display(pd.DataFrame(rows).T.round(4))

,ndcg_10,recall_100,recall_1000
RRF only (no GBDT),0.3053,0.5908,0.7960
judged only,0.2865,0.5010,0.7175
judged + 100 random,0.3156,0.5904,0.8042
current: 50 top + 50 random,0.1683,0.2598,0.5672
"current, rerank RRF top 1000",0.2720,0.4383,0.7960
"judged only, rerank top 1000",0.2869,0.4963,0.7960


In [27]:

# ---- Demo: search both engines side by side ------------------------------------------------------
import html
import ipywidgets as W

TEXT_TO_QID = {t: q for q, t in query_text.items()}
LABEL_NAME = {v: k for k, v in LABEL_INT.items()}
BADGE = {"E": "#1a7f37", "S": "#0969da", "C": "#bc4c00", "I": "#cf222e", None: "#8c959f"}
ASINS = products["product_id"].to_numpy()
rng_demo = np.random.default_rng()


def labels_of(text, ids):
    qid = TEXT_TO_QID.get(text)
    if qid is None:
        return [None] * len(ids)
    lab, found = labels_for(qid, np.asarray(ids, dtype=np.int64))
    return [LABEL_NAME[int(l)] if f else None for l, f in zip(lab, found)]


def badge(lab):
    return (f"<span style='background:{BADGE[lab]};color:#fff;border-radius:3px;padding:0 5px;"
            f"font-size:11px;font-weight:600'>{lab or '–'}</span>")


def chips(facets):
    out = []
    for f in FACET_FIELDS:
        items = list(facets.get(f, {}).items())[:6]
        out.append(f"<div style='margin:2px 0'><b>{f}</b>: " +
                   ", ".join(f"{html.escape(str(k))} ({v:,})" for k, v in items) + "</div>")
    return "".join(out)


q_box = W.Text(value=TEST.texts[0], placeholder="Search products…", continuous_update=False,
               layout=W.Layout(width="420px"))
go_b = W.Button(description="Search", button_style="primary")
rand_b = W.Button(description="Random test query")
flt_dd = W.Dropdown(options=[("No filter", None)], value=None, layout=W.Layout(width="360px"))
prev_b, next_b = W.Button(description="◀ Prev"), W.Button(description="Next ▶")
status = W.HTML()
out = W.HTML()
state = {"page": 0, "busy": False}


def render(fresh):
    text = q_box.value.strip()
    if not text:
        out.value = "<i>Type a query.</i>"
        return
    flt, cols, judged_known = flt_dd.value, [], text in TEXT_TO_QID
    for eng in ENGINES:
        tm, t0 = Timer(), time.perf_counter()
        res = page(eng, text, state["page"], flt, use_cache=not fresh, timer=tm)
        ms = (time.perf_counter() - t0) * 1000
        if fresh:
            top = sorted(((k, v) for k, v in tm.laps.items() if k != "page"), key=lambda kv: -kv[1])[:3]
            timing = f"<b>{ms:.1f} ms</b> · " + " · ".join(f"{k} {v:.1f}" for k, v in top)
        else:
            timing = f"<b>{ms:.2f} ms</b> (served from page cache)"
        labs = labels_of(text, res["ids"])
        counts = collections.Counter(labs)
        rows = []
        for rank, (doc, title, lab) in enumerate(zip(res["ids"], res["titles"], labs), state["page"] * PAGE_SIZE + 1):
            meta = " · ".join(x for x in (FACET["brand"]["values"][doc], FACET["color"]["values"][doc]) if x)
            rows.append(f"<div style='padding:4px 0;border-bottom:1px solid #8883'>"
                        f"<span style='color:#8c959f;width:28px;display:inline-block'>{rank}</span>{badge(lab)} "
                        f"<a href='https://www.amazon.com/dp/{ASINS[doc]}' target='_blank'>{html.escape(title[:90])}</a>"
                        f"<div style='margin-left:28px;color:#8c959f;font-size:11px'>{html.escape(meta)}</div></div>")
        summary = (" · ".join(f"{k} {counts.get(k, 0)}" for k in "ESCI") + f" · unjudged {counts.get(None, 0)}"
                   if judged_known else "not an ESCI query, so no judgments")
        mode = "GBDT" if eng.ranker is not None else "RRF order"
        cols.append(f"<div style='flex:1;min-width:300px;border:1px solid #8885;border-radius:8px;padding:10px'>"
                    f"<div style='font-weight:700;font-size:14px'>{html.escape(eng.name)} "
                    f"<span style='font-weight:400;color:#8c959f'>({mode})</span></div>"
                    f"<div style='font-size:12px;margin:4px 0'>{timing}</div>"
                    f"<div style='font-size:12px'>{res['total']:,} matches · page labels: {summary}</div>"
                    f"<div style='font-size:12px;margin:6px 0'>{chips(res['facets'])}</div>{''.join(rows)}</div>")
        if fresh and eng is GPU_FULL and flt is None:
            opts = [("No filter", None)] + [(f"{f}: {k} ({v:,})", (f, k))
                                            for f in FACET_FIELDS for k, v in list(res["facets"].get(f, {}).items())[:10]]
            state["busy"] = True
            flt_dd.options, flt_dd.value = opts, None
            state["busy"] = False
    status.value = (f"Query <b>{html.escape(text)}</b> · filter <b>{html.escape(str(flt[1]) if flt else 'none')}</b> · "
                    f"page {state['page'] + 1} · {PAGE_SIZE} per page · badges: E exact, S substitute, C complement, I irrelevant")
    out.value = f"<div style='display:flex;gap:10px;flex-wrap:wrap;font-family:sans-serif'>{''.join(cols)}</div>"


def search(*_):
    if state["busy"]:
        return
    state["page"] = 0
    render(fresh=True)


def on_filter(change):
    if not state["busy"]:
        state["page"] = 0
        render(fresh=True)


def turn(delta):
    state["page"] = max(0, min(K_FINAL // PAGE_SIZE - 1, state["page"] + delta))
    render(fresh=False)


def random_query(_):
    state["busy"] = True
    q_box.value = TEST.texts[int(rng_demo.integers(len(TEST)))]
    flt_dd.options, flt_dd.value = [("No filter", None)], None
    state["busy"] = False
    search()


go_b.on_click(search)
q_box.observe(lambda c: search() if c["name"] == "value" else None, names="value")
flt_dd.observe(on_filter, names="value")
prev_b.on_click(lambda _: turn(-1))
next_b.on_click(lambda _: turn(+1))
rand_b.on_click(random_query)

display(W.VBox([W.HBox([q_box, go_b, rand_b]), W.HBox([flt_dd, prev_b, next_b]), status, out]))
search()

In [ ]:
# ---- Save what the standalone demo notebook needs to Google Drive ----------------------------------
import shutil
from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = Path("/content/drive/MyDrive/esci_demo")
SAVE_OPENSEARCH = True    # The index is several GB. Without it, the demo rebuilds it (about 30 minutes).
SAVE_DIR.mkdir(parents=True, exist_ok=True)


def ranker_file(name):
    return "ranker_" + re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_").lower() + ".json"


demo_state = dict(cache_id=cache_id, os_index=OS_INDEX, os_version=OS_VERSION, os_pipe_name=OS_PIPE_NAME,
                  os_tuned=[OS_TUNED[0], OS_TUNED[1]], embed_model=EMBED_MODEL,
                  bm25f_tuned=dict(w=list(BM25F_TUNED.w), b=list(BM25F_TUNED.b), k1=BM25F_TUNED.k1),
                  rankers={e.name: ranker_file(e.name) for e in ENGINES})
(SAVE_DIR / "demo_state.json").write_text(json.dumps(demo_state, indent=2, default=float))
for eng in ENGINES:
    eng.ranker.booster.save_model(str(SAVE_DIR / ranker_file(eng.name)))
print("Saved tuned settings and", len(ENGINES), "rankers")

for src in (PRODUCTS_FILE, EXAMPLES_FILE, postings_file, emb_file):
    dst = SAVE_DIR / src.name
    if src.resolve() == dst.resolve() or (dst.exists() and dst.stat().st_size == src.stat().st_size):
        continue
    t0 = time.perf_counter()
    shutil.copyfile(src, dst)
    print(f"Copied {src.name} ({src.stat().st_size / 2**30:.2f} GB) in {time.perf_counter() - t0:.0f}s")

if SAVE_OPENSEARCH:
    # Stop OpenSearch cleanly, archive the installation with its data, then start it again.
    client.indices.flush(index=OS_INDEX, request_timeout=600)
    subprocess.run(["pkill", "-f", "org.opensearch.bootstrap.OpenSearch"])
    for _ in range(90):
        if subprocess.run(["pgrep", "-f", "org.opensearch.bootstrap.OpenSearch"], capture_output=True).returncode != 0:
            break
        time.sleep(2)
    t0 = time.perf_counter()
    tar_path = SAVE_DIR / "opensearch.tar"
    subprocess.run(["tar", "-cf", str(tar_path), "--exclude", f"opensearch-{OS_VERSION}/logs",
                    "-C", "/content", f"opensearch-{OS_VERSION}"], check=True)
    print(f"Saved OpenSearch with its index ({tar_path.stat().st_size / 2**30:.1f} GB) in {time.perf_counter() - t0:.0f}s")
    start_opensearch()

drive.flush_and_unmount()   # Makes sure every file has finished uploading
print("Done. Open esci_demo.ipynb in a new GPU runtime and choose Run all.")

Mounted at /content/drive
Saved tuned settings and 3 rankers
Copied shopping_queries_dataset_products.parquet (1.03 GB) in 3s
Copied shopping_queries_dataset_examples.parquet (0.05 GB) in 0s
Copied postings_17101d00f1.pkl (0.94 GB) in 6s
Copied emb_BAAI_bge_small_en_v1_5_17101d00f1.npy (0.87 GB) in 2s


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/opensearchpy/connection/http_urllib3.py", line 280, in perform_request
    response = self.pool.urlopen(
        method, url, body, retries=Retry(False), headers=request_headers, **kw
    )
  File "/usr/local/lib/python3.13/dist-packages/urlli

Saved OpenSearch with its index (6.0 GB) in 50s


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/opensearchpy/connection/http_urllib3.py", line 280, in perform_request
    response = self.pool.urlopen(
        method, url, body, retries=Retry(False), headers=request_headers, **kw
    )
  File "/usr/local/lib/python3.13/dist-packages/urlli

OpenSearch is up: 2.19.1
Done. Open esci_demo.ipynb in a new GPU runtime and choose Run all.
